# 🔬 NAFNet H2 — Kaggle Pipeline

Physically-motivated compound-degradation image restoration.  
Scroll to the **Configuration** cell, update your data paths, then **Run All**.

| # | Section | Notes |
|---|---------|-------|
| 1 | **Setup** | pip installs, NAFNet GitHub clone |
| 2 | **Source Code** | Writes all `src/` modules to disk |
| 3 | **Configuration** | ← **Edit paths & hyperparams here** |
| 4 | **Dataset Check** | Verifies paired `.npy` files |
| 5 | **Pretrained Weights** | Optional auto-download from NAFNet releases |
| 6 | **Training** | Full pipeline — DC loss, SSIM ramp, curriculum aug, auto-resume |
| 7 | **Training Curves** | Plots `history.json` |
| 8 | **Inference** | TTA prediction on test set |
| 9 | **Submission** | Creates `submission.csv` |

> **Multi-GPU:** The training cell auto-detects GPUs and uses `torchrun` when > 1 is available.


In [ ]:
import subprocess, sys

pkgs = ["pytorch-msssim", "lpips", "tqdm"]
for pkg in pkgs:
    r = subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"],
                       capture_output=True, text=True)
    print(f"  {'✓' if r.returncode == 0 else '✗'} {pkg}")
print("\nDependency install done.")


In [ ]:
import os, subprocess, sys

NAFNET_DIR = "nafnet/official"

if not os.path.isdir(NAFNET_DIR):
    os.makedirs("nafnet", exist_ok=True)
    print("Cloning megvii-research/NAFNet …")
    r = subprocess.run(
        ["git", "clone", "--depth=1",
         "https://github.com/megvii-research/NAFNet.git", NAFNET_DIR],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print(r.stderr)
        raise RuntimeError("NAFNet clone failed. Check network access.")
    print("  Cloned successfully.")
else:
    print(f"NAFNet already present: {NAFNET_DIR}")

for p in [NAFNET_DIR, "."]:
    if p not in sys.path:
        sys.path.insert(0, p)

for d in ["src", "outputs/nafnet_h2", "nafnet/weights"]:
    os.makedirs(d, exist_ok=True)
print("Directories ready.")


In [ ]:
%%writefile src/__init__.py
# src package — written by notebook setup


In [ ]:
%%writefile src/config.py
from __future__ import annotations

import argparse
import json
from pathlib import Path
from typing import Any


def _load_yaml(path: Path) -> dict[str, Any]:
    try:
        import yaml  # type: ignore
    except Exception as exc:  # pragma: no cover
        raise RuntimeError(
            "YAML config requested but PyYAML is not installed. Install with: pip install pyyaml"
        ) from exc

    with path.open("r", encoding="utf-8") as f:
        data = yaml.safe_load(f)
    return data or {}


def load_config_file(path: str) -> dict[str, Any]:
    if not path:
        return {}

    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"Config file not found: {path}")

    suffix = p.suffix.lower()
    if suffix in {".yaml", ".yml"}:
        data = _load_yaml(p)
    elif suffix == ".json":
        with p.open("r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        raise ValueError("Config file must be .yaml/.yml or .json")

    if not isinstance(data, dict):
        raise ValueError("Config root must be a dictionary")
    return data


def _collect_leaf_keys(obj: Any, out: dict[str, Any]) -> None:
    if isinstance(obj, dict):
        for key, value in obj.items():
            if isinstance(value, dict):
                _collect_leaf_keys(value, out)
            else:
                out[str(key)] = value


def apply_config_defaults(parser: argparse.ArgumentParser, config_data: dict[str, Any]) -> dict[str, Any]:
    if not config_data:
        return {}

    flat: dict[str, Any] = {}
    _collect_leaf_keys(config_data, flat)

    known_dests = {action.dest for action in parser._actions}
    to_apply = {k: v for k, v in flat.items() if k in known_dests}
    unknown = {k: v for k, v in flat.items() if k not in known_dests}

    if to_apply:
        parser.set_defaults(**to_apply)

    return unknown


In [ ]:
%%writefile src/ddp.py
from __future__ import annotations

import os
import subprocess
from typing import Dict

import torch
import torch.distributed as dist


def init_distributed_mode() -> dict:
    rank = int(os.environ.get("RANK", "0"))
    world_size = int(os.environ.get("WORLD_SIZE", "1"))
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))

    distributed = world_size > 1

    if torch.cuda.is_available():
        torch.cuda.set_device(local_rank)
        device = torch.device("cuda", local_rank)
        backend = "nccl"
    else:
        device = torch.device("cpu")
        backend = "gloo"

    if distributed and not dist.is_initialized():
        dist.init_process_group(backend=backend, init_method="env://", rank=rank, world_size=world_size)

    return {
        "rank": rank,
        "world_size": world_size,
        "local_rank": local_rank,
        "distributed": distributed,
        "device": device,
    }


def is_main_process(rank: int) -> bool:
    return rank == 0


def barrier(distributed: bool) -> None:
    if distributed and dist.is_initialized():
        dist.barrier()


def cleanup_distributed(distributed: bool) -> None:
    if distributed and dist.is_initialized():
        dist.destroy_process_group()


def reduce_mean(value: torch.Tensor, distributed: bool) -> torch.Tensor:
    if not distributed:
        return value
    value = value.clone()
    dist.all_reduce(value, op=dist.ReduceOp.SUM)
    value /= dist.get_world_size()
    return value


def reduce_dict(metrics: Dict[str, float], device: torch.device, distributed: bool) -> Dict[str, float]:
    if not distributed:
        return metrics

    keys = sorted(metrics.keys())
    vals = torch.tensor([metrics[k] for k in keys], device=device, dtype=torch.float32)
    dist.all_reduce(vals, op=dist.ReduceOp.SUM)
    vals /= dist.get_world_size()
    return {k: float(v) for k, v in zip(keys, vals.tolist())}


def cuda_environment_summary() -> dict:
    nvidia_smi_gpus = []
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=index,name", "--format=csv,noheader"],
            stderr=subprocess.DEVNULL,
            text=True,
        )
        for line in out.strip().splitlines():
            idx_str, name = [x.strip() for x in line.split(",", 1)]
            nvidia_smi_gpus.append({"index": int(idx_str), "name": name})
    except Exception:
        nvidia_smi_gpus = []

    if not torch.cuda.is_available():
        return {
            "cuda": False,
            "torch_gpu_count": int(torch.cuda.device_count()),
            "torch_gpus": [],
            "nvidia_smi_gpus": nvidia_smi_gpus,
        }

    gpus = []
    for idx in range(torch.cuda.device_count()):
        gpus.append({"index": idx, "name": torch.cuda.get_device_name(idx)})

    return {
        "cuda": True,
        "torch_gpu_count": len(gpus),
        "torch_gpus": gpus,
        "nvidia_smi_gpus": nvidia_smi_gpus,
    }


In [ ]:
%%writefile src/dataset.py
from __future__ import annotations

import os
import random
from dataclasses import dataclass
from typing import List, Sequence, Tuple

import numpy as np
import torch
from torch.utils.data import Dataset


Pair = Tuple[str, str, str]


def _to_chw(arr: np.ndarray) -> np.ndarray:
    if arr.ndim == 2:
        return arr[None, ...]
    if arr.ndim == 3:
        if arr.shape[0] in (1, 3):
            return arr
        if arr.shape[-1] in (1, 3):
            return np.transpose(arr, (2, 0, 1))
    raise ValueError(f"Unsupported ndarray shape: {arr.shape}")


def load_npy_image(path: str) -> np.ndarray:
    arr = np.load(path).astype(np.float32)
    return _to_chw(arr)


def list_train_pairs(gt_dir: str, noisy_dir: str) -> List[Pair]:
    gt_names = {f for f in os.listdir(gt_dir) if f.endswith(".npy")}
    noisy_names = {f for f in os.listdir(noisy_dir) if f.endswith(".npy")}
    common = sorted(gt_names & noisy_names)

    pairs: List[Pair] = []
    for name in common:
        pairs.append((name, os.path.join(gt_dir, name), os.path.join(noisy_dir, name)))
    return pairs


def split_pairs(pairs: Sequence[Pair], val_ratio: float = 0.02, seed: int = 42) -> Tuple[List[Pair], List[Pair]]:
    if not 0.0 <= val_ratio < 1.0:
        raise ValueError("val_ratio must be in [0, 1)")

    idxs = list(range(len(pairs)))
    rng = random.Random(seed)
    rng.shuffle(idxs)

    n_val = int(round(len(pairs) * val_ratio))
    val_idxs = set(idxs[:n_val])

    train_pairs = [pairs[i] for i in range(len(pairs)) if i not in val_idxs]
    val_pairs = [pairs[i] for i in range(len(pairs)) if i in val_idxs]
    return train_pairs, val_pairs


def _spatially_align(gt: np.ndarray, noisy: np.ndarray, scale: int) -> Tuple[np.ndarray, np.ndarray]:
    h_gt, w_gt = gt.shape[-2:]
    h_lr, w_lr = noisy.shape[-2:]

    h_lr_aligned = min(h_lr, h_gt // scale)
    w_lr_aligned = min(w_lr, w_gt // scale)

    gt = gt[..., : h_lr_aligned * scale, : w_lr_aligned * scale]
    noisy = noisy[..., :h_lr_aligned, :w_lr_aligned]
    return gt, noisy


def _random_crop_pair(gt: np.ndarray, noisy: np.ndarray, patch_size: int, scale: int, enforce_lr_multiple: int = 8) -> Tuple[np.ndarray, np.ndarray]:
    h_gt, w_gt = gt.shape[-2:]

    hr_patch = min(patch_size, h_gt, w_gt)
    base_multiple = max(scale * enforce_lr_multiple, scale)
    hr_patch = (hr_patch // base_multiple) * base_multiple
    if hr_patch == 0:
        hr_patch = (min(h_gt, w_gt) // scale) * scale
    if hr_patch <= 0:
        raise ValueError("Patch size became zero after alignment. Use larger input images.")

    lr_patch = hr_patch // scale
    h_lr, w_lr = noisy.shape[-2:]

    if h_lr == lr_patch:
        top_lr = 0
    else:
        top_lr = random.randint(0, h_lr - lr_patch)

    if w_lr == lr_patch:
        left_lr = 0
    else:
        left_lr = random.randint(0, w_lr - lr_patch)

    top_gt = top_lr * scale
    left_gt = left_lr * scale

    gt = gt[..., top_gt : top_gt + hr_patch, left_gt : left_gt + hr_patch]
    noisy = noisy[..., top_lr : top_lr + lr_patch, left_lr : left_lr + lr_patch]
    return gt, noisy


def _blur3x3_reflect(x: np.ndarray) -> np.ndarray:
    pad_w = np.pad(x, ((0, 0), (0, 0), (1, 1)), mode="reflect")
    tmp = (pad_w[:, :, :-2] + 2.0 * pad_w[:, :, 1:-1] + pad_w[:, :, 2:]) * 0.25

    pad_h = np.pad(tmp, ((0, 0), (1, 1), (0, 0)), mode="reflect")
    out = (pad_h[:, :-2, :] + 2.0 * pad_h[:, 1:-1, :] + pad_h[:, 2:, :]) * 0.25
    return out.astype(np.float32, copy=False)


def _clip_noisy(x: np.ndarray, clip_min: float, clip_max: float) -> np.ndarray:
    return np.clip(x, clip_min, clip_max).astype(np.float32, copy=False)


def _synthetic_ood_degrade(
    noisy: np.ndarray,
    profile: str,
    max_transforms: int,
    clip_min: float,
    clip_max: float,
) -> np.ndarray:
    if profile == "basic":
        return noisy

    c, h, w = noisy.shape
    if h <= 2 or w <= 2:
        return noisy

    def _op_affine(v: np.ndarray) -> np.ndarray:
        gain = random.uniform(0.75, 1.25)
        bias = random.uniform(-0.08, 0.08)
        return v * gain + bias

    def _op_gamma(v: np.ndarray) -> np.ndarray:
        base = np.clip(v, 0.0, 1.0)
        gamma = random.uniform(0.7, 1.5)
        mapped = np.power(base + 1e-6, gamma)
        blend = random.uniform(0.4, 0.9)
        return blend * mapped + (1.0 - blend) * base

    def _op_gaussian_noise(v: np.ndarray) -> np.ndarray:
        sigma = random.uniform(0.005, 0.06 if profile == "max" else 0.04)
        return v + np.random.normal(0.0, sigma, size=v.shape).astype(np.float32)

    def _op_speckle(v: np.ndarray) -> np.ndarray:
        sigma = random.uniform(0.01, 0.08 if profile == "max" else 0.05)
        return v * (1.0 + np.random.normal(0.0, sigma, size=v.shape).astype(np.float32))

    def _op_poisson(v: np.ndarray) -> np.ndarray:
        peak = random.uniform(20.0, 120.0)
        base = np.clip(v, 0.0, 1.0)
        sampled = np.random.poisson(base * peak).astype(np.float32) / peak
        blend = random.uniform(0.5, 0.9)
        return blend * sampled + (1.0 - blend) * base

    def _op_blur(v: np.ndarray) -> np.ndarray:
        out = _blur3x3_reflect(v)
        if random.random() < 0.4:
            out = _blur3x3_reflect(out)
        return out

    def _op_stripe(v: np.ndarray) -> np.ndarray:
        amp = random.uniform(0.005, 0.04 if profile == "max" else 0.025)
        if random.random() < 0.5:
            stripe = np.random.normal(0.0, amp, size=(1, h, 1)).astype(np.float32)
        else:
            stripe = np.random.normal(0.0, amp, size=(1, 1, w)).astype(np.float32)
        return v + stripe

    def _op_impulse(v: np.ndarray) -> np.ndarray:
        ratio = random.uniform(0.001, 0.02 if profile == "max" else 0.008)
        mask = np.random.rand(1, h, w) < ratio
        vals = np.random.uniform(clip_min, clip_max, size=(1, h, w)).astype(np.float32)
        return np.where(mask, vals, v)

    def _op_cutout(v: np.ndarray) -> np.ndarray:
        out = v.copy()
        holes = random.randint(1, 3 if profile == "max" else 2)
        for _ in range(holes):
            hh = random.randint(max(4, h // 20), max(8, h // 5))
            ww = random.randint(max(4, w // 20), max(8, w // 5))
            top = random.randint(0, max(h - hh, 0))
            left = random.randint(0, max(w - ww, 0))
            fill = float(np.random.uniform(clip_min, clip_max))
            out[:, top : top + hh, left : left + ww] = fill
        return out

    ops = [_op_affine, _op_gamma, _op_gaussian_noise, _op_speckle, _op_poisson, _op_blur, _op_stripe, _op_impulse, _op_cutout]

    if profile == "max":
        n_ops = random.randint(2, max(2, max_transforms))
    else:
        n_ops = random.randint(1, max(1, min(max_transforms, 3)))

    order = random.sample(ops, k=min(n_ops, len(ops)))
    out = noisy.astype(np.float32, copy=False)
    for op in order:
        out = _clip_noisy(op(out), clip_min=clip_min, clip_max=clip_max)
    return out


def _augment_pair(
    gt: np.ndarray,
    noisy: np.ndarray,
    augment_profile: str,
    ood_prob: float,
    ood_max_transforms: int,
    ood_clip_min: float,
    ood_clip_max: float,
) -> Tuple[np.ndarray, np.ndarray]:
    if random.random() < 0.5:
        gt = np.flip(gt, axis=-1)
        noisy = np.flip(noisy, axis=-1)

    if random.random() < 0.5:
        gt = np.flip(gt, axis=-2)
        noisy = np.flip(noisy, axis=-2)

    k = random.randint(0, 3)
    if k:
        gt = np.rot90(gt, k=k, axes=(-2, -1))
        noisy = np.rot90(noisy, k=k, axes=(-2, -1))

    if augment_profile != "basic" and random.random() < ood_prob:
        noisy = _synthetic_ood_degrade(
            noisy,
            profile=augment_profile,
            max_transforms=ood_max_transforms,
            clip_min=ood_clip_min,
            clip_max=ood_clip_max,
        )

    return gt.copy(), noisy.copy()


@dataclass(frozen=True)
class TrainDatasetConfig:
    scale: int = 2
    patch_size: int = 256
    augment: bool = True
    augment_profile: str = "basic"  # basic | strong | max
    ood_prob: float = 0.0
    ood_max_transforms: int = 4
    ood_clip_min: float = -0.25
    ood_clip_max: float = 1.80


class NpyPairDataset(Dataset):
    """Paired GT/NoisyLR dataset loaded from .npy files."""

    def __init__(self, pairs: Sequence[Pair], cfg: TrainDatasetConfig, training: bool) -> None:
        self.pairs = list(pairs)
        self.cfg = cfg
        self.training = training
        self.augment_profile = cfg.augment_profile
        self.ood_prob = float(cfg.ood_prob)
        self.ood_max_transforms = int(cfg.ood_max_transforms)
        self.ood_clip_min = float(cfg.ood_clip_min)
        self.ood_clip_max = float(cfg.ood_clip_max)

    def set_ood_policy(self, profile: str, prob: float, max_transforms: int) -> None:
        self.augment_profile = str(profile)
        self.ood_prob = float(max(0.0, min(1.0, prob)))
        self.ood_max_transforms = int(max(1, max_transforms))

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, index: int) -> dict:
        name, gt_path, noisy_path = self.pairs[index]
        gt = load_npy_image(gt_path)
        noisy = load_npy_image(noisy_path)

        gt, noisy = _spatially_align(gt, noisy, scale=self.cfg.scale)

        if self.training and self.cfg.patch_size > 0:
            gt, noisy = _random_crop_pair(gt, noisy, patch_size=self.cfg.patch_size, scale=self.cfg.scale)

        if self.training and self.cfg.augment:
            gt, noisy = _augment_pair(
                gt,
                noisy,
                augment_profile=self.augment_profile,
                ood_prob=self.ood_prob,
                ood_max_transforms=self.ood_max_transforms,
                ood_clip_min=self.ood_clip_min,
                ood_clip_max=self.ood_clip_max,
            )

        # GT is normalized in [0,1] by dataset definition, but clipping adds numerical safety.
        gt = np.clip(gt, 0.0, 1.0)

        return {
            "name": name,
            "gt": torch.from_numpy(np.ascontiguousarray(gt)),
            "noisy": torch.from_numpy(np.ascontiguousarray(noisy)),
        }


class NpyNoisyDataset(Dataset):
    """Noisy-only dataset for inference; expects .npy tensors in LR space."""

    def __init__(self, noisy_dir: str) -> None:
        self.noisy_dir = noisy_dir
        self.names = sorted([f for f in os.listdir(noisy_dir) if f.endswith(".npy")])

    def __len__(self) -> int:
        return len(self.names)

    def __getitem__(self, index: int) -> dict:
        name = self.names[index]
        path = os.path.join(self.noisy_dir, name)
        noisy = load_npy_image(path)
        return {
            "name": name,
            "noisy": torch.from_numpy(np.ascontiguousarray(noisy)),
        }


In [ ]:
%%writefile src/losses.py
from __future__ import annotations

import math
from dataclasses import dataclass

import torch
import torch.nn.functional as F

try:
    from pytorch_msssim import ssim as _msssim  # type: ignore[import-not-found]

    HAS_MSSSIM = True
except Exception:
    _msssim = None
    HAS_MSSSIM = False

try:
    import lpips as _lpips_pkg  # type: ignore[import-not-found]

    HAS_LPIPS = True
except Exception:
    _lpips_pkg = None
    HAS_LPIPS = False


@dataclass
class LossConfig:
    scale: int = 2
    h2_a: float = 0.14160734
    h2_b: float = 6.3383e-05
    student_t_nu: float = 9.0
    lambda_psnr: float = 1.0
    lambda_l1: float = 0.0
    lambda_l2: float = 0.0
    lambda_charbonnier: float = 0.0
    lambda_fft: float = 0.0
    lambda_ssim: float = 0.10
    lambda_dc: float = 0.05
    ssim_contribute_to_loss: bool = True
    lambda_lpips: float = 0.0
    pixel_loss_type: str = "l1"
    charbonnier_eps: float = 1e-3
    lambda_edge: float = 0.0


def _as_tensor(value: float | torch.Tensor, ref: torch.Tensor) -> torch.Tensor:
    if torch.is_tensor(value):
        return value.to(device=ref.device, dtype=ref.dtype)
    return torch.tensor(value, device=ref.device, dtype=ref.dtype)


def forward_consistency_h2(x_hat: torch.Tensor, scale: int = 2) -> tuple[torch.Tensor, torch.Tensor]:
    """
    H2 forward branch from inferred pipeline:
      mu = D(x_hat)
      q  = D(x_hat^2) / (scale^2)
    where D is area-like downsampling (avg pooling here).
    """

    mu = F.avg_pool2d(x_hat, kernel_size=scale, stride=scale)
    q = F.avg_pool2d(x_hat * x_hat, kernel_size=scale, stride=scale) / float(scale * scale)
    return mu, q


def heteroscedastic_variance_h2(q: torch.Tensor, h2_a: float | torch.Tensor, h2_b: float | torch.Tensor) -> torch.Tensor:
    a = _as_tensor(h2_a, q)
    b = _as_tensor(h2_b, q)
    return torch.clamp(b + a * q, min=1e-8)


def student_t_nll(y: torch.Tensor, mu: torch.Tensor, variance: torch.Tensor, nu: float = 9.0) -> torch.Tensor:
    if nu <= 2.0:
        raise ValueError("student_t_nu must be > 2")

    variance = torch.clamp(variance, min=1e-8)
    resid2 = (y - mu) ** 2

    nu_t = torch.tensor(nu, dtype=variance.dtype, device=variance.device)
    log_norm = torch.lgamma((nu_t + 1.0) / 2.0) - torch.lgamma(nu_t / 2.0) - 0.5 * torch.log(nu_t * torch.tensor(torch.pi, device=variance.device, dtype=variance.dtype))

    log_pdf = (
        log_norm
        - 0.5 * torch.log(variance)
        - 0.5 * (nu_t + 1.0) * torch.log1p(resid2 / (nu_t * variance))
    )
    return -log_pdf.mean()


def dc_loss_robust(y: torch.Tensor, mu: torch.Tensor, variance: torch.Tensor, nu: float = 9.0) -> torch.Tensor:
    """
    Robust Student-t-style data term without log(v), using detached variance weights.

    This prevents the model from exploiting variance-collapse pathways while retaining
    heavy-tail robustness in LR reprojection space.
    """

    if nu <= 0.0:
        raise ValueError("student_t_nu must be > 0")

    resid2 = (y - mu) ** 2
    v_w = torch.clamp(variance.detach(), min=1e-4)
    return torch.log1p(resid2 / (nu * v_w)).mean()


def ssim_loss(x_hat: torch.Tensor, x_gt: torch.Tensor) -> torch.Tensor:
    if not HAS_MSSSIM or _msssim is None:
        return torch.zeros((), device=x_hat.device, dtype=x_hat.dtype)
    x_hat = x_hat.clamp(0.0, 1.0)
    x_gt = x_gt.clamp(0.0, 1.0)
    return 1.0 - _msssim(x_hat, x_gt, data_range=1.0, size_average=True)


def build_lpips_model(net: str = "alex", device: torch.device | None = None) -> torch.nn.Module | None:
    if not HAS_LPIPS or _lpips_pkg is None:
        return None

    model = _lpips_pkg.LPIPS(net=net, verbose=False)
    if device is not None:
        model = model.to(device)
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)
    return model


def lpips_loss(x_hat: torch.Tensor, x_gt: torch.Tensor, lpips_net: torch.nn.Module | None) -> torch.Tensor:
    if lpips_net is None:
        return torch.zeros((), device=x_hat.device, dtype=x_hat.dtype)

    x_hat = x_hat.clamp(0.0, 1.0)
    x_gt = x_gt.clamp(0.0, 1.0)

    # LPIPS expects RGB tensors in [-1, 1].
    x_hat = x_hat * 2.0 - 1.0
    x_gt = x_gt * 2.0 - 1.0
    if x_hat.shape[1] == 1:
        x_hat = x_hat.repeat(1, 3, 1, 1)
        x_gt = x_gt.repeat(1, 3, 1, 1)

    score = lpips_net(x_hat, x_gt)
    return score.mean()


def charbonnier_loss(x_hat: torch.Tensor, x_gt: torch.Tensor, eps: float = 1e-3) -> torch.Tensor:
    if eps <= 0:
        raise ValueError("charbonnier_eps must be > 0")
    diff = x_hat - x_gt
    eps_t = torch.tensor(eps, device=diff.device, dtype=diff.dtype)
    return torch.sqrt(diff * diff + eps_t * eps_t).mean()


def edge_loss(x_hat: torch.Tensor, x_gt: torch.Tensor) -> torch.Tensor:
    grad_x_hat = x_hat[:, :, :, 1:] - x_hat[:, :, :, :-1]
    grad_x_gt = x_gt[:, :, :, 1:] - x_gt[:, :, :, :-1]
    grad_y_hat = x_hat[:, :, 1:, :] - x_hat[:, :, :-1, :]
    grad_y_gt = x_gt[:, :, 1:, :] - x_gt[:, :, :-1, :]
    return F.l1_loss(grad_x_hat, grad_x_gt) + F.l1_loss(grad_y_hat, grad_y_gt)


def fourier_magnitude_loss(x_hat: torch.Tensor, x_gt: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    if eps <= 0:
        raise ValueError("eps must be > 0")

    x_hat_fft = torch.fft.rfft2(x_hat, norm="ortho")
    x_gt_fft = torch.fft.rfft2(x_gt, norm="ortho")
    mag_hat = torch.abs(x_hat_fft)
    mag_gt = torch.abs(x_gt_fft)

    # Log compression emphasizes relative structure and stabilizes bright-frequency dominance.
    return F.l1_loss(torch.log(mag_hat + eps), torch.log(mag_gt + eps))


def psnr_loss_official(x_hat: torch.Tensor, x_gt: torch.Tensor) -> torch.Tensor:
        """
        Match official NAFNet PSNRLoss objective:
            (10 / ln(10)) * ln(MSE + 1e-8)
        averaged across batch.
        """

        mse = ((x_hat - x_gt) ** 2).mean(dim=(1, 2, 3))
        scale = 10.0 / math.log(10.0)
        return scale * torch.log(mse + 1e-8).mean()


def combined_restoration_loss(
    x_hat: torch.Tensor,
    x_gt: torch.Tensor,
    y_lr: torch.Tensor,
    cfg: LossConfig,
    use_dc: bool,
    lpips_net: torch.nn.Module | None = None,
) -> tuple[torch.Tensor, dict]:
    l_psnr = psnr_loss_official(x_hat, x_gt)
    l_l1 = F.l1_loss(x_hat, x_gt)
    l_l2 = F.mse_loss(x_hat, x_gt)
    l_charb = charbonnier_loss(x_hat, x_gt, eps=cfg.charbonnier_eps)
    l_fft = fourier_magnitude_loss(x_hat, x_gt)

    weighted_recon = (
        cfg.lambda_psnr * l_psnr
        + cfg.lambda_l1 * l_l1
        + cfg.lambda_l2 * l_l2
        + cfg.lambda_charbonnier * l_charb
    )

    # Backward compatibility: if no mixed recon weights are provided, use legacy one-of pixel_loss_type.
    if (cfg.lambda_psnr + cfg.lambda_l1 + cfg.lambda_l2 + cfg.lambda_charbonnier) > 0:
        recon = weighted_recon
    elif cfg.pixel_loss_type == "charbonnier":
        recon = l_charb
    elif cfg.pixel_loss_type == "psnr":
        recon = l_psnr
    else:
        recon = l_l1

    l_ssim = ssim_loss(x_hat, x_gt)
    if cfg.lambda_lpips > 0:
        l_lpips = lpips_loss(x_hat, x_gt, lpips_net=lpips_net)
    else:
        l_lpips = torch.zeros((), device=x_hat.device, dtype=x_hat.dtype)
    if cfg.lambda_edge > 0:
        l_edge = edge_loss(x_hat, x_gt)
    else:
        l_edge = torch.zeros((), device=x_hat.device, dtype=x_hat.dtype)

    if use_dc:
        mu, q = forward_consistency_h2(x_hat, scale=cfg.scale)
        v = heteroscedastic_variance_h2(q, h2_a=cfg.h2_a, h2_b=cfg.h2_b)
        l_dc = dc_loss_robust(y_lr, mu, v, nu=cfg.student_t_nu)
    else:
        l_dc = torch.zeros((), device=x_hat.device, dtype=x_hat.dtype)

    total = recon + cfg.lambda_fft * l_fft + cfg.lambda_dc * l_dc
    if cfg.ssim_contribute_to_loss:
        total = total + cfg.lambda_ssim * l_ssim
    if cfg.lambda_lpips > 0:
        total = total + cfg.lambda_lpips * l_lpips
    if cfg.lambda_edge > 0:
        total = total + cfg.lambda_edge * l_edge

    logs = {
        "loss_total": float(total.detach().item()),
        "loss_recon": float(recon.detach().item()),
        "loss_psnr": float(l_psnr.detach().item()),
        "loss_l1_metric": float(l_l1.detach().item()),
        "loss_l1": float(l_l1.detach().item()),
        "loss_l2": float(l_l2.detach().item()),
        "loss_charbonnier": float(l_charb.detach().item()),
        "loss_fft": float(l_fft.detach().item()),
        "loss_ssim": float(l_ssim.detach().item()),
        "loss_dc": float(l_dc.detach().item()),
        "loss_lpips": float(l_lpips.detach().item()),
        "loss_edge": float(l_edge.detach().item()),
        "lambda_psnr": float(cfg.lambda_psnr),
        "lambda_l1": float(cfg.lambda_l1),
        "lambda_l2": float(cfg.lambda_l2),
        "lambda_charbonnier": float(cfg.lambda_charbonnier),
        "lambda_fft": float(cfg.lambda_fft),
        "lambda_dc": float(cfg.lambda_dc),
        "lambda_ssim": float(cfg.lambda_ssim),
        "lambda_lpips": float(cfg.lambda_lpips),
        "lambda_edge": float(cfg.lambda_edge),
        "pixel_loss_type": cfg.pixel_loss_type,
    }
    return total, logs


In [ ]:
%%writefile src/model.py
from __future__ import annotations

import importlib
import os
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Sequence

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass(frozen=True)
class NAFNetH2Config:
    in_channels: int = 1
    out_channels: int = 1
    scale: int = 2
    width: int = 64
    enc_blk_nums: Sequence[int] = (2, 2, 4, 8)
    middle_blk_num: int = 12
    dec_blk_nums: Sequence[int] = (2, 2, 2, 2)
    official_repo: str = "nafnet/official"
    upsample_mode: str = "bilinear"
    clamp_output: bool = True


def _as_int_tuple(v: Sequence[int]) -> tuple[int, ...]:
    return tuple(int(x) for x in v)


def _resolve_state_dict(payload: object) -> tuple[str, dict]:
    if isinstance(payload, dict):
        if "params_ema" in payload and isinstance(payload["params_ema"], dict):
            return "params_ema", payload["params_ema"]
        if "params" in payload and isinstance(payload["params"], dict):
            return "params", payload["params"]
        if "model" in payload and isinstance(payload["model"], dict):
            return "model", payload["model"]
        if payload and all(isinstance(k, str) for k in payload.keys()):
            if any(torch.is_tensor(v) for v in payload.values()):
                return "root", payload
    raise ValueError("Could not locate a model state dict in checkpoint payload")


def _ensure_official_repo_on_path(official_repo: str) -> Path:
    repo = Path(official_repo).resolve()
    if not repo.exists():
        raise FileNotFoundError(
            f"Official NAFNet repo not found at {repo}. "
            "Clone it first: git clone https://github.com/megvii-research/NAFNet.git nafnet/official"
        )
    repo_str = str(repo)
    if repo_str not in sys.path:
        sys.path.insert(0, repo_str)
    return repo


def _import_official_nafnet(official_repo: str):
    _ensure_official_repo_on_path(official_repo)
    module = importlib.import_module("basicsr.models.archs.NAFNet_arch")
    if not hasattr(module, "NAFNet"):
        raise RuntimeError("Failed to import NAFNet from official repository")
    return getattr(module, "NAFNet")


class NAFNetH2SR(nn.Module):
    """
    Wrapper around official NAFNet for LR noisy -> HR restored.

    Pipeline:
      1) Upsample LR input to HR with interpolation.
      2) Replicate grayscale to RGB for compatibility with official pretrained RGB NAFNet.
      3) Run official NAFNet.
      4) Collapse RGB prediction to grayscale output.
    """

    def __init__(self, cfg: NAFNetH2Config) -> None:
        super().__init__()
        if cfg.scale <= 0:
            raise ValueError("scale must be > 0")
        if cfg.upsample_mode not in {"nearest", "bilinear", "bicubic"}:
            raise ValueError("upsample_mode must be one of: nearest/bilinear/bicubic")

        self.cfg = cfg
        self.scale = int(cfg.scale)
        self.in_channels = int(cfg.in_channels)
        self.out_channels = int(cfg.out_channels)
        self.upsample_mode = cfg.upsample_mode
        self.clamp_output = bool(cfg.clamp_output)

        nafnet_cls = _import_official_nafnet(cfg.official_repo)
        self.backbone = nafnet_cls(
            img_channel=3,
            width=int(cfg.width),
            middle_blk_num=int(cfg.middle_blk_num),
            enc_blk_nums=_as_int_tuple(cfg.enc_blk_nums),
            dec_blk_nums=_as_int_tuple(cfg.dec_blk_nums),
        )

    def _upsample(self, y: torch.Tensor) -> torch.Tensor:
        if self.scale == 1:
            return y
        if self.upsample_mode == "nearest":
            return F.interpolate(y, scale_factor=self.scale, mode=self.upsample_mode)
        return F.interpolate(y, scale_factor=self.scale, mode=self.upsample_mode, align_corners=False)

    def forward(self, y: torch.Tensor) -> torch.Tensor:
        if y.ndim != 4:
            raise ValueError(f"Expected 4D tensor [B,C,H,W], got shape: {tuple(y.shape)}")

        y_up = self._upsample(y)

        if y_up.shape[1] == 1:
            x = y_up.repeat(1, 3, 1, 1)
        elif y_up.shape[1] == 3:
            x = y_up
        else:
            raise ValueError("NAFNetH2SR expects input channels to be 1 or 3")

        out_rgb = self.backbone(x)

        if self.out_channels == 1:
            out = out_rgb.mean(dim=1, keepdim=True)
        elif self.out_channels == 3:
            out = out_rgb
        else:
            raise ValueError("NAFNetH2SR currently supports out_channels=1 or 3")

        if self.clamp_output:
            out = out.clamp(0.0, 1.0)

        h_lr, w_lr = y.shape[-2:]
        return out[:, :, : h_lr * self.scale, : w_lr * self.scale]


def make_nafnet_config(
    preset: str,
    in_channels: int = 1,
    out_channels: int = 1,
    scale: int = 2,
    official_repo: str = "nafnet/official",
    upsample_mode: str = "bilinear",
) -> NAFNetH2Config:
    p = preset.lower()
    if p in {"width32", "sidd-width32", "nafnet-sidd-width32"}:
        return NAFNetH2Config(
            in_channels=in_channels,
            out_channels=out_channels,
            scale=scale,
            width=32,
            enc_blk_nums=(2, 2, 4, 8),
            middle_blk_num=12,
            dec_blk_nums=(2, 2, 2, 2),
            official_repo=official_repo,
            upsample_mode=upsample_mode,
        )
    if p in {"width64", "sidd-width64", "nafnet-sidd-width64"}:
        return NAFNetH2Config(
            in_channels=in_channels,
            out_channels=out_channels,
            scale=scale,
            width=64,
            enc_blk_nums=(2, 2, 4, 8),
            middle_blk_num=12,
            dec_blk_nums=(2, 2, 2, 2),
            official_repo=official_repo,
            upsample_mode=upsample_mode,
        )
    raise ValueError(f"Unknown NAFNet preset: {preset}")


def load_nafnet_pretrained(model: NAFNetH2SR, checkpoint_path: str, strict: bool = False) -> dict:
    p = Path(checkpoint_path)
    if not p.exists():
        raise FileNotFoundError(f"Pretrained checkpoint not found: {p}")

    payload = torch.load(p, map_location="cpu")
    source, raw_state = _resolve_state_dict(payload)

    state = {}
    for k, v in raw_state.items():
        nk = k
        if nk.startswith("module."):
            nk = nk[len("module.") :]
        if nk.startswith("backbone."):
            nk = nk[len("backbone.") :]
        state[nk] = v

    missing, unexpected = model.backbone.load_state_dict(state, strict=strict)
    matched = len(state) - len(unexpected)

    return {
        "checkpoint": str(p),
        "source": source,
        "strict": bool(strict),
        "matched": int(max(matched, 0)),
        "missing": int(len(missing)),
        "unexpected": int(len(unexpected)),
        "missing_keys": list(missing),
        "unexpected_keys": list(unexpected),
    }


In [ ]:
%%writefile src/train_nafnet_ddp.py
from __future__ import annotations

import argparse
import json
import math
import os
import random
import shutil
import time
import warnings
from dataclasses import replace
from pathlib import Path

import numpy as np
import torch
import torch.distributed as dist
import torch.nn.functional as F
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.utils.data import DataLoader, DistributedSampler
from tqdm import tqdm

from src.model import NAFNetH2SR, load_nafnet_pretrained, make_nafnet_config
from src.config import apply_config_defaults, load_config_file
from src.dataset import NpyPairDataset, TrainDatasetConfig, list_train_pairs, split_pairs
from src.ddp import (
    barrier,
    cleanup_distributed,
    cuda_environment_summary,
    init_distributed_mode,
    is_main_process,
    reduce_dict,
)
from src.losses import (
    HAS_LPIPS,
    HAS_MSSSIM,
    LossConfig,
    build_lpips_model,
    combined_restoration_loss,
    forward_consistency_h2,
    heteroscedastic_variance_h2,
    lpips_loss,
    ssim_loss,
)


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description="Train official NAFNet (author implementation) on H2 data under DDP")

    parser.add_argument("--config", type=str, default="", help="Path to YAML/JSON config file")

    parser.add_argument("--data-root", type=str, default="../data/train")
    parser.add_argument("--gt-subdir", type=str, default="GT")
    parser.add_argument("--noisy-subdir", type=str, default="NoisyLR")
    parser.add_argument("--output-dir", type=str, default="outputs/nafnet_h2")

    parser.add_argument("--nafnet-preset", type=str, default="sidd-width64", choices=["sidd-width32", "sidd-width64"])
    parser.add_argument("--official-repo", type=str, default="nafnet/official")
    parser.add_argument("--pretrained", type=str, default="nafnet/weights/nafnet_sidd_width64.pth")
    parser.add_argument("--strict-pretrained", action=argparse.BooleanOptionalAction, default=False)
    parser.add_argument(
        "--staged-freeze",
        action=argparse.BooleanOptionalAction,
        default=False,
        help="Apply progressive freezing of early encoder stages during finetuning",
    )
    parser.add_argument(
        "--freeze-stage1-end-epoch",
        type=int,
        default=10,
        help="Epoch where phase-1 freezing ends (phase-1 active for epoch < value)",
    )
    parser.add_argument(
        "--freeze-stage2-end-epoch",
        type=int,
        default=25,
        help="Epoch where phase-2 partial freeze ends (phase-2 active for epoch < value)",
    )
    parser.add_argument(
        "--freeze-intro",
        action=argparse.BooleanOptionalAction,
        default=True,
        help="Keep NAFNet intro stem frozen during staged-freeze phases",
    )
    parser.add_argument("--in-channels", type=int, default=1)
    parser.add_argument("--scale", type=int, default=2)
    parser.add_argument("--upsample-mode", type=str, default="bilinear", choices=["nearest", "bilinear", "bicubic"])

    parser.add_argument("--epochs", type=int, default=140)
    parser.add_argument("--batch-size", type=int, default=8)
    parser.add_argument("--num-workers", type=int, default=8)
    parser.add_argument("--lr", type=float, default=1e-3)
    parser.add_argument("--weight-decay", type=float, default=0.0)
    parser.add_argument("--grad-clip", type=float, default=0.0)
    parser.add_argument("--optim-beta1", type=float, default=0.9)
    parser.add_argument("--optim-beta2", type=float, default=0.9)
    parser.add_argument("--amp", action=argparse.BooleanOptionalAction, default=True)
    parser.add_argument("--lr-warmup-epochs", type=int, default=0)
    parser.add_argument("--lr-warmup-start-factor", type=float, default=0.05)
    parser.add_argument("--lr-eta-min", type=float, default=1e-7)
    parser.add_argument(
        "--total-iters",
        type=int,
        default=400000,
        help="Total optimizer updates for cosine schedule; set <=0 to use epochs*steps_per_epoch.",
    )

    parser.add_argument("--patch-size", type=int, default=256)
    parser.add_argument("--augment-profile", type=str, default="max", choices=["basic", "strong", "max"])
    parser.add_argument("--ood-prob", type=float, default=0.95)
    parser.add_argument("--ood-max-transforms", type=int, default=5)
    parser.add_argument("--ood-clip-min", type=float, default=-0.25)
    parser.add_argument("--ood-clip-max", type=float, default=1.8)
    parser.add_argument("--augment-curriculum", action=argparse.BooleanOptionalAction, default=True)
    parser.add_argument("--val-ratio", type=float, default=0.10)
    parser.add_argument("--seed", type=int, default=42)

    parser.add_argument("--finetune-preset", type=str, default="roi_psnr_120e", choices=["none", "roi_psnr_120e"])
    parser.add_argument("--stage1-end-epoch", type=int, default=20)
    parser.add_argument("--stage2-end-epoch", type=int, default=90)

    parser.add_argument("--lambda-psnr", type=float, default=1.0)
    parser.add_argument("--lambda-l1", type=float, default=0.0)
    parser.add_argument("--lambda-l2", type=float, default=0.0)
    parser.add_argument("--lambda-charbonnier", type=float, default=0.0)
    parser.add_argument("--lambda-fft", type=float, default=0.0)

    parser.add_argument("--warmup-epochs", type=int, default=10)
    parser.add_argument("--dc-schedule", type=str, default="constant", choices=["adaptive", "linear", "constant"])
    parser.add_argument("--dc-lambda-start", type=float, default=0.005)
    parser.add_argument("--dc-lambda-step", type=float, default=0.005)
    parser.add_argument("--dc-lambda-cap", type=float, default=0.04)
    parser.add_argument("--dc-patience", type=int, default=2)
    parser.add_argument("--dc-min-delta", type=float, default=1e-4)
    parser.add_argument("--dc-ramp-epochs", type=int, default=10)

    parser.add_argument("--lambda-ssim", type=float, default=0.05)
    parser.add_argument("--ssim-start-epoch", type=int, default=60)
    parser.add_argument("--ssim-lambda-start", type=float, default=0.0005)
    parser.add_argument("--ssim-ramp-epochs", type=int, default=100)
    parser.add_argument("--ssim-contribute-to-loss", action=argparse.BooleanOptionalAction, default=False)
    parser.add_argument("--pixel-loss-type", type=str, default="psnr", choices=["l1", "charbonnier", "psnr"])
    parser.add_argument("--charbonnier-eps", type=float, default=1e-3)
    parser.add_argument("--lambda-edge", type=float, default=0.0)

    parser.add_argument("--lambda-dc", type=float, default=0.03)
    parser.add_argument("--student-t-nu", type=float, default=3.0)

    parser.add_argument("--lambda-lpips-max", type=float, default=0.0)
    parser.add_argument("--lpips-start-epoch", type=int, default=60)
    parser.add_argument("--lpips-lambda-start", type=float, default=0.001)
    parser.add_argument("--lpips-ramp-epochs", type=int, default=30)
    parser.add_argument("--lpips-gate-psnr", type=float, default=25.5)
    parser.add_argument("--lpips-gate-min-epoch", type=int, default=40)
    parser.add_argument("--lpips-net", type=str, default="alex", choices=["alex", "vgg"])

    parser.add_argument("--require-ssim", action=argparse.BooleanOptionalAction, default=True)
    parser.add_argument("--require-lpips", action=argparse.BooleanOptionalAction, default=False)
    parser.add_argument("--dc-debug-first-epoch", action=argparse.BooleanOptionalAction, default=False)

    parser.add_argument("--h2-a", type=float, default=0.14160734)
    parser.add_argument("--h2-b", type=float, default=6.3383e-05)
    parser.add_argument("--h2-jitter", type=float, default=0.025)
    parser.add_argument("--h2-jitter-decay", type=str, default="none", choices=["none", "linear", "cosine"])
    parser.add_argument("--h2-jitter-min", type=float, default=0.025)

    parser.add_argument("--resume", type=str, default="")
    parser.add_argument("--auto-resume", action=argparse.BooleanOptionalAction, default=True)
    parser.add_argument(
        "--save-every",
        type=int,
        default=5,
        help="Save periodic checkpoint every N epochs; set <=0 to disable periodic saves.",
    )
    parser.add_argument("--export-best-infer", action=argparse.BooleanOptionalAction, default=True)
    parser.add_argument("--export-latest-infer", action=argparse.BooleanOptionalAction, default=True)

    return parser


def parse_args() -> argparse.Namespace:
    pre = argparse.ArgumentParser(add_help=False)
    pre.add_argument("--config", type=str, default="")
    known, _ = pre.parse_known_args()

    parser = build_parser()
    if known.config:
        cfg_data = load_config_file(known.config)
        unknown = apply_config_defaults(parser, cfg_data)
        if unknown:
            print(f"[Config] Ignored unknown keys: {sorted(list(unknown.keys()))}")

    return parser.parse_args()


def resolve_resume_path(args: argparse.Namespace, out_dir: Path) -> Path | None:
    if args.resume:
        p = Path(args.resume)
        if not p.exists():
            raise FileNotFoundError(f"Resume checkpoint not found: {p}")
        return p

    if args.auto_resume:
        latest = out_dir / "latest.pt"
        if latest.exists():
            return latest

    return None


class DCScheduler:
    def __init__(
        self,
        start: float = 0.005,
        step: float = 0.005,
        cap: float = 0.05,
        patience: int = 2,
        min_delta: float = 1e-4,
    ) -> None:
        self.lam = float(max(0.0, min(start, cap)))
        self.step = float(step)
        self.cap = float(cap)
        self.patience = int(patience)
        self.min_delta = float(max(0.0, min_delta))

        self._no_improve = 0
        self._best_psnr = -1e9

    def update(self, val_psnr: float) -> float:
        if val_psnr > self._best_psnr + self.min_delta:
            self._best_psnr = float(val_psnr)
            self._no_improve = 0
            self.lam = min(self.cap, self.lam + self.step)
        else:
            self._no_improve += 1
            if self._no_improve >= self.patience:
                self.lam = max(self.step, self.lam - self.step)
                self._no_improve = 0
        return self.lam

    def state_dict(self) -> dict:
        return {
            "lam": self.lam,
            "step": self.step,
            "cap": self.cap,
            "patience": self.patience,
            "min_delta": self.min_delta,
            "_no_improve": self._no_improve,
            "_best_psnr": self._best_psnr,
        }

    def load_state_dict(self, state: dict) -> None:
        if not state:
            return
        self.lam = float(state.get("lam", self.lam))
        self.step = float(state.get("step", self.step))
        self.cap = float(state.get("cap", self.cap))
        self.patience = int(state.get("patience", self.patience))
        self.min_delta = float(state.get("min_delta", self.min_delta))
        self._no_improve = int(state.get("_no_improve", self._no_improve))
        self._best_psnr = float(state.get("_best_psnr", self._best_psnr))


def build_infer_export_payload(save_payload: dict) -> dict:
    return {
        "epoch": save_payload["epoch"],
        "model": save_payload["model"],
        "best_psnr": save_payload.get("best_psnr", -1e9),
        "best_lpips": save_payload.get("best_lpips", float("inf")),
        "args": save_payload.get("args", {}),
    }


def seed_everything(seed: int, rank: int) -> None:
    s = seed + rank
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)


def effective_h2_jitter(args: argparse.Namespace, epoch: int) -> float:
    base = float(max(args.h2_jitter, 0.0))
    if base <= 0.0:
        return 0.0
    if args.h2_jitter_decay == "none":
        return base

    if args.epochs <= 1:
        return max(float(args.h2_jitter_min), base)

    progress = min(max(epoch / float(args.epochs - 1), 0.0), 1.0)
    if args.h2_jitter_decay == "cosine":
        factor = 0.5 * (1.0 + math.cos(math.pi * progress))
    else:
        factor = 1.0 - progress

    return max(float(args.h2_jitter_min), base * factor)


def effective_lambda_ssim(args: argparse.Namespace, epoch: int) -> float:
    if epoch < args.ssim_start_epoch:
        return 0.0

    lam_cap = float(max(args.lambda_ssim, 0.0))
    lam_start = float(max(args.ssim_lambda_start, 0.0))
    lam_start = min(lam_start, lam_cap)

    if args.ssim_ramp_epochs <= 0:
        return lam_cap

    progress = min(max((epoch - args.ssim_start_epoch) / float(args.ssim_ramp_epochs), 0.0), 1.0)
    return lam_start + (lam_cap - lam_start) * progress


def effective_lambda_lpips(args: argparse.Namespace, epoch: int, lpips_gate_epoch: int | None) -> float:
    lam_cap = float(max(args.lambda_lpips_max, 0.0))
    if lam_cap <= 0.0:
        return 0.0
    if lpips_gate_epoch is None or epoch < lpips_gate_epoch:
        return 0.0

    lam_start = float(max(args.lpips_lambda_start, 0.0))
    lam_start = min(lam_start, lam_cap)

    if args.lpips_ramp_epochs <= 0:
        return lam_cap

    progress = min(max((epoch - lpips_gate_epoch) / float(args.lpips_ramp_epochs), 0.0), 1.0)
    return lam_start + (lam_cap - lam_start) * progress


def _lerp(a: float, b: float, t: float) -> float:
    return a + (b - a) * t


def _interp_triplet(epoch: int, e1: int, e2: int, e_end: int, a: float, b: float, c: float) -> float:
    if epoch < e1:
        return float(a)
    if epoch < e2:
        t = (epoch - e1) / float(max(e2 - e1, 1))
        return float(_lerp(a, b, t))

    tail = max(e_end - e2 - 1, 1)
    t = min(max((epoch - e2) / float(tail), 0.0), 1.0)
    return float(_lerp(b, c, t))


def effective_stage_weights(args: argparse.Namespace, epoch: int, max_epochs: int) -> dict[str, float]:
    if args.finetune_preset != "roi_psnr_120e":
        return {
            "lambda_psnr": float(args.lambda_psnr),
            "lambda_l1": float(args.lambda_l1),
            "lambda_l2": float(args.lambda_l2),
            "lambda_charbonnier": float(args.lambda_charbonnier),
            "lambda_fft": float(args.lambda_fft),
            "lambda_dc": float(args.lambda_dc),
            "lambda_ssim": float(args.lambda_ssim),
        }

    e1 = max(int(args.stage1_end_epoch), 0)
    e2 = max(int(args.stage2_end_epoch), e1)
    e_end = max(int(max_epochs), e2 + 1)

    return {
        "lambda_psnr": _interp_triplet(epoch, e1, e2, e_end, 1.0, 1.0, 1.0),
        "lambda_l1": _interp_triplet(epoch, e1, e2, e_end, 0.05, 0.03, 0.02),
        "lambda_l2": _interp_triplet(epoch, e1, e2, e_end, 0.01, 0.005, 0.0),
        "lambda_charbonnier": _interp_triplet(epoch, e1, e2, e_end, 0.01, 0.008, 0.005),
        "lambda_fft": _interp_triplet(epoch, e1, e2, e_end, 0.0, 0.01, 0.015),
        "lambda_dc": _interp_triplet(epoch, e1, e2, e_end, 0.0, 0.01, 0.008),
        "lambda_ssim": _interp_triplet(epoch, e1, e2, e_end, 0.0, 0.015, 0.02),
    }


def effective_augment_policy(args: argparse.Namespace, epoch: int, max_epochs: int) -> dict[str, float | int | str]:
    max_prob = float(max(0.0, min(1.0, args.ood_prob)))
    max_ops = int(max(1, args.ood_max_transforms))
    profile = str(args.augment_profile)

    if not args.augment_curriculum:
        return {
            "profile": profile,
            "ood_prob": max_prob,
            "ood_max_transforms": max_ops,
        }

    e1 = max(int(args.stage1_end_epoch), 0)
    e2 = max(int(args.stage2_end_epoch), e1)
    e_end = max(int(max_epochs), e2 + 1)

    if epoch < e1:
        return {
            "profile": "strong" if profile == "max" else profile,
            "ood_prob": min(max_prob, 0.70),
            "ood_max_transforms": min(max_ops, 3),
        }

    if epoch < e2:
        return {
            "profile": profile,
            "ood_prob": max_prob,
            "ood_max_transforms": max_ops,
        }

    tail = max(e_end - e2 - 1, 1)
    t = min(max((epoch - e2) / float(tail), 0.0), 1.0)
    prob_start = min(max_prob, 0.80)
    prob_end = min(max_prob, 0.50)
    ops_start = min(max_ops, 3)
    ops_end = min(max_ops, 2)

    return {
        "profile": "strong" if profile == "max" else profile,
        "ood_prob": _lerp(prob_start, prob_end, t),
        "ood_max_transforms": int(round(_lerp(float(ops_start), float(ops_end), t))),
    }


@torch.no_grad()
def evaluate(
    model: torch.nn.Module,
    loader: DataLoader,
    device: torch.device,
    distributed: bool,
    lpips_net: torch.nn.Module | None,
) -> dict:
    model.eval()

    count = 0
    loss_l1 = 0.0
    loss_mse = 0.0
    metric_ssim = 0.0
    metric_lpips = 0.0

    for batch in loader:
        y = batch["noisy"].to(device, non_blocking=True)
        x = batch["gt"].to(device, non_blocking=True)

        pred = model(y)
        l1 = F.l1_loss(pred, x)
        mse = F.mse_loss(pred, x)
        ssim_score = 1.0 - ssim_loss(pred, x) if HAS_MSSSIM else torch.zeros((), device=device, dtype=pred.dtype)
        lpips_score = lpips_loss(pred, x, lpips_net=lpips_net)

        bsz = y.shape[0]
        count += bsz
        loss_l1 += float(l1.item()) * bsz
        loss_mse += float(mse.item()) * bsz
        metric_ssim += float(ssim_score.item()) * bsz
        metric_lpips += float(lpips_score.item()) * bsz

    metrics = {
        "val_l1": loss_l1 / max(count, 1),
        "val_mse": loss_mse / max(count, 1),
        "val_ssim": metric_ssim / max(count, 1),
        "val_lpips": metric_lpips / max(count, 1),
    }
    metrics["val_psnr"] = float(10.0 * np.log10(1.0 / max(metrics["val_mse"], 1e-12)))
    return reduce_dict(metrics, device=device, distributed=distributed)


def make_loader(
    dataset,
    batch_size: int,
    num_workers: int,
    distributed: bool,
    rank: int,
    world_size: int,
    shuffle: bool,
    drop_last: bool,
    device: torch.device,
    persistent_workers: bool,
):
    sampler = None
    if distributed:
        sampler = DistributedSampler(
            dataset,
            num_replicas=world_size,
            rank=rank,
            shuffle=shuffle,
            drop_last=drop_last,
        )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=(sampler is None and shuffle),
        sampler=sampler,
        num_workers=num_workers,
        pin_memory=(device.type == "cuda"),
        drop_last=drop_last,
        persistent_workers=(persistent_workers and num_workers > 0),
    )
    return loader, sampler


def _all_ranks_finite(t: torch.Tensor, distributed: bool) -> bool:
    finite_local = torch.tensor(1 if torch.isfinite(t).all() else 0, device=t.device, dtype=torch.int32)
    if distributed:
        dist.all_reduce(finite_local, op=dist.ReduceOp.MIN)
    return bool(finite_local.item() == 1)


def _set_module_trainable(module: torch.nn.Module | None, trainable: bool) -> None:
    if module is None:
        return
    for p in module.parameters():
        p.requires_grad_(trainable)


def _freeze_phase_for_epoch(args: argparse.Namespace, epoch: int) -> str:
    if not args.staged_freeze:
        return "all_trainable"
    if epoch < args.freeze_stage1_end_epoch:
        return "freeze_early"
    if epoch < args.freeze_stage2_end_epoch:
        return "freeze_partial"
    return "all_trainable"


def _apply_freeze_phase(model: torch.nn.Module, args: argparse.Namespace, phase: str) -> tuple[int, int]:
    core = model.module if isinstance(model, DDP) else model
    backbone = getattr(core, "backbone", None)
    if backbone is None:
        total = sum(p.numel() for p in core.parameters())
        trainable = sum(p.numel() for p in core.parameters() if p.requires_grad)
        return trainable, total

    # Reset to fully trainable first, then freeze selected stages for current phase.
    _set_module_trainable(backbone, True)
    if phase == "freeze_early":
        if args.freeze_intro:
            _set_module_trainable(getattr(backbone, "intro", None), False)
        encoders = getattr(backbone, "encoders", None)
        downs = getattr(backbone, "downs", None)
        if encoders is not None:
            if len(encoders) > 0:
                _set_module_trainable(encoders[0], False)
            if len(encoders) > 1:
                _set_module_trainable(encoders[1], False)
        if downs is not None:
            if len(downs) > 0:
                _set_module_trainable(downs[0], False)
            if len(downs) > 1:
                _set_module_trainable(downs[1], False)
    elif phase == "freeze_partial":
        if args.freeze_intro:
            _set_module_trainable(getattr(backbone, "intro", None), False)
        encoders = getattr(backbone, "encoders", None)
        downs = getattr(backbone, "downs", None)
        if encoders is not None and len(encoders) > 0:
            _set_module_trainable(encoders[0], False)
        if downs is not None and len(downs) > 0:
            _set_module_trainable(downs[0], False)

    total = sum(p.numel() for p in core.parameters())
    trainable = sum(p.numel() for p in core.parameters() if p.requires_grad)
    return trainable, total


def main() -> None:
    args = parse_args()

    warnings.filterwarnings(
        "ignore",
        message=r"The epoch parameter in `scheduler.step\(\)` was not necessary and is being deprecated where possible.*",
        category=UserWarning,
        module=r"torch\.optim\.lr_scheduler",
    )

    if args.freeze_stage1_end_epoch < 0:
        raise ValueError("freeze_stage1_end_epoch must be >= 0")
    if args.freeze_stage2_end_epoch < 0:
        raise ValueError("freeze_stage2_end_epoch must be >= 0")
    if args.freeze_stage2_end_epoch < args.freeze_stage1_end_epoch:
        raise ValueError("freeze_stage2_end_epoch must be >= freeze_stage1_end_epoch")

    if args.ssim_start_epoch < 0:
        raise ValueError("ssim_start_epoch must be >= 0")
    if args.ssim_ramp_epochs < 0:
        raise ValueError("ssim_ramp_epochs must be >= 0")
    if args.ssim_lambda_start < 0.0:
        raise ValueError("ssim_lambda_start must be >= 0")
    if args.lambda_ssim > 0.0 and args.ssim_lambda_start > args.lambda_ssim:
        raise ValueError("ssim_lambda_start must be <= lambda_ssim")
    if args.charbonnier_eps <= 0:
        raise ValueError("charbonnier_eps must be > 0")
    if args.lambda_edge < 0:
        raise ValueError("lambda_edge must be >= 0")
    if args.lambda_lpips_max < 0.0:
        raise ValueError("lambda_lpips_max must be >= 0")
    if args.lpips_start_epoch < 0:
        raise ValueError("lpips_start_epoch must be >= 0")
    if args.lpips_gate_min_epoch < 0:
        raise ValueError("lpips_gate_min_epoch must be >= 0")
    if args.lpips_ramp_epochs < 0:
        raise ValueError("lpips_ramp_epochs must be >= 0")
    if args.lpips_lambda_start < 0.0:
        raise ValueError("lpips_lambda_start must be >= 0")
    if args.lpips_lambda_start > args.lambda_lpips_max:
        raise ValueError("lpips_lambda_start must be <= lambda_lpips_max")
    if args.h2_jitter_min < 0.0:
        raise ValueError("h2_jitter_min must be >= 0")
    if args.h2_jitter_min > args.h2_jitter:
        raise ValueError("h2_jitter_min must be <= h2_jitter")
    if not (0.0 <= args.optim_beta1 < 1.0):
        raise ValueError("optim_beta1 must be in [0, 1)")
    if not (0.0 <= args.optim_beta2 < 1.0):
        raise ValueError("optim_beta2 must be in [0, 1)")
    if args.lr_eta_min < 0.0:
        raise ValueError("lr_eta_min must be >= 0")
    if not (0.0 <= args.ood_prob <= 1.0):
        raise ValueError("ood_prob must be in [0, 1]")
    if args.ood_max_transforms <= 0:
        raise ValueError("ood_max_transforms must be > 0")
    if args.ood_clip_min >= args.ood_clip_max:
        raise ValueError("ood_clip_min must be < ood_clip_max")
    if args.stage1_end_epoch < 0:
        raise ValueError("stage1_end_epoch must be >= 0")
    if args.stage2_end_epoch < args.stage1_end_epoch:
        raise ValueError("stage2_end_epoch must be >= stage1_end_epoch")
    for name in ("lambda_psnr", "lambda_l1", "lambda_l2", "lambda_charbonnier", "lambda_fft"):
        if float(getattr(args, name)) < 0.0:
            raise ValueError(f"{name} must be >= 0")

    env = init_distributed_mode()
    rank = env["rank"]
    world_size = env["world_size"]
    local_rank = env["local_rank"]
    distributed = env["distributed"]
    device = env["device"]

    seed_everything(args.seed, rank)

    out_dir = Path(args.output_dir)
    if is_main_process(rank):
        out_dir.mkdir(parents=True, exist_ok=True)

    barrier(distributed)

    if is_main_process(rank):
        print("CUDA environment:", json.dumps(cuda_environment_summary(), indent=2))
        print(f"Distributed: {distributed}, world_size={world_size}, local_rank={local_rank}")

    if args.lambda_ssim > 0 and not HAS_MSSSIM:
        msg = "lambda_ssim > 0 but pytorch_msssim is not available. Install with: pip install pytorch-msssim"
        if args.require_ssim:
            raise RuntimeError(msg)
        if is_main_process(rank):
            print(f"[Warning] {msg}. SSIM term will be zero.")

    gt_dir = os.path.join(args.data_root, args.gt_subdir)
    noisy_dir = os.path.join(args.data_root, args.noisy_subdir)
    pairs = list_train_pairs(gt_dir, noisy_dir)
    if len(pairs) == 0:
        raise RuntimeError(f"No paired .npy files found in {gt_dir} and {noisy_dir}")

    train_pairs, val_pairs = split_pairs(pairs, val_ratio=args.val_ratio, seed=args.seed)

    train_cfg = TrainDatasetConfig(
        scale=args.scale,
        patch_size=args.patch_size,
        augment=True,
        augment_profile=args.augment_profile,
        ood_prob=args.ood_prob,
        ood_max_transforms=args.ood_max_transforms,
        ood_clip_min=args.ood_clip_min,
        ood_clip_max=args.ood_clip_max,
    )
    val_cfg = TrainDatasetConfig(scale=args.scale, patch_size=args.patch_size, augment=False, augment_profile="basic")

    train_ds = NpyPairDataset(train_pairs, cfg=train_cfg, training=True)
    val_ds = NpyPairDataset(val_pairs, val_cfg, training=False)

    train_persistent_workers = bool(args.num_workers > 0 and not args.augment_curriculum)
    if is_main_process(rank) and args.augment_curriculum and args.num_workers > 0:
        print("[Augment] Curriculum enabled: disabling persistent train workers for epoch-wise policy updates.")

    train_loader, train_sampler = make_loader(
        train_ds,
        batch_size=args.batch_size,
        num_workers=args.num_workers,
        distributed=distributed,
        rank=rank,
        world_size=world_size,
        shuffle=True,
        drop_last=True,
        device=device,
        persistent_workers=train_persistent_workers,
    )

    val_loader, val_sampler = make_loader(
        val_ds,
        batch_size=max(1, args.batch_size // 2),
        num_workers=max(1, args.num_workers // 2),
        distributed=distributed,
        rank=rank,
        world_size=world_size,
        shuffle=False,
        drop_last=False,
        device=device,
        persistent_workers=(args.num_workers > 0),
    )

    model_cfg = make_nafnet_config(
        preset=args.nafnet_preset,
        in_channels=args.in_channels,
        out_channels=args.in_channels,
        scale=args.scale,
        official_repo=args.official_repo,
        upsample_mode=args.upsample_mode,
    )
    model = NAFNetH2SR(model_cfg).to(device)

    resume_path = resolve_resume_path(args, out_dir)
    pretrained_report = None
    if resume_path is None and args.pretrained:
        pretrained_report = load_nafnet_pretrained(model, args.pretrained, strict=args.strict_pretrained)
        if is_main_process(rank):
            print("Loaded official pretrained:", json.dumps(pretrained_report, indent=2))

    if distributed:
        model = DDP(
            model,
            device_ids=[local_rank] if device.type == "cuda" else None,
            find_unused_parameters=bool(args.staged_freeze),
        )

    optimizer = AdamW(
        model.parameters(),
        lr=args.lr,
        weight_decay=args.weight_decay,
        betas=(args.optim_beta1, args.optim_beta2),
    )

    steps_per_epoch = max(len(train_loader), 1)
    target_total_iters = int(args.total_iters) if args.total_iters > 0 else int(args.epochs * steps_per_epoch)
    if target_total_iters <= 0:
        raise ValueError("target_total_iters must be > 0")

    warmup_iters = max(int(args.lr_warmup_epochs * steps_per_epoch), 0)
    warmup_iters = min(warmup_iters, max(target_total_iters - 1, 0))

    if warmup_iters > 0:
        scheduler = SequentialLR(
            optimizer,
            schedulers=[
                LinearLR(optimizer, start_factor=args.lr_warmup_start_factor, end_factor=1.0, total_iters=warmup_iters),
                CosineAnnealingLR(
                    optimizer,
                    T_max=max(target_total_iters - warmup_iters, 1),
                    eta_min=args.lr_eta_min,
                ),
            ],
            milestones=[warmup_iters],
        )
    else:
        scheduler = CosineAnnealingLR(optimizer, T_max=max(target_total_iters, 1), eta_min=args.lr_eta_min)
    scaler = torch.amp.GradScaler(device=device.type, enabled=(args.amp and device.type == "cuda"))

    lpips_net = None
    if args.lambda_lpips_max > 0:
        msg = "lambda_lpips_max > 0 but lpips is not available. Install with: pip install lpips"
        if not HAS_LPIPS:
            if args.require_lpips:
                raise RuntimeError(msg)
            if is_main_process(rank):
                print(f"[Warning] {msg}. LPIPS term will be zero.")
        else:
            lpips_net = build_lpips_model(net=args.lpips_net, device=device)
            if lpips_net is None and args.require_lpips:
                raise RuntimeError(msg)
            if is_main_process(rank):
                print(f"LPIPS objective enabled with net='{args.lpips_net}'.")
    lpips_enabled = lpips_net is not None and args.lambda_lpips_max > 0

    loss_cfg = LossConfig(
        scale=args.scale,
        h2_a=args.h2_a,
        h2_b=args.h2_b,
        student_t_nu=args.student_t_nu,
        lambda_psnr=args.lambda_psnr,
        lambda_l1=args.lambda_l1,
        lambda_l2=args.lambda_l2,
        lambda_charbonnier=args.lambda_charbonnier,
        lambda_fft=args.lambda_fft,
        lambda_ssim=args.lambda_ssim,
        lambda_dc=args.lambda_dc,
        ssim_contribute_to_loss=args.ssim_contribute_to_loss,
        lambda_lpips=0.0,
        pixel_loss_type=args.pixel_loss_type,
        charbonnier_eps=args.charbonnier_eps,
        lambda_edge=args.lambda_edge,
    )

    dc_schedule_mode = args.dc_schedule
    dc_sched = None
    if dc_schedule_mode == "adaptive":
        dc_sched = DCScheduler(
            start=args.dc_lambda_start,
            step=args.dc_lambda_step,
            cap=args.dc_lambda_cap,
            patience=args.dc_patience,
            min_delta=args.dc_min_delta,
        )

    start_epoch = 0
    global_step = 0
    best_psnr = -1e9
    best_lpips = float("inf")
    lpips_gate_epoch: int | None = None
    prev_val_psnr: float | None = None

    if resume_path is not None:
        ckpt = torch.load(resume_path, map_location="cpu")
        target = model.module if isinstance(model, DDP) else model
        target.load_state_dict(ckpt["model"], strict=True)
        optimizer.load_state_dict(ckpt["optimizer"])
        try:
            scheduler.load_state_dict(ckpt["scheduler"])
        except Exception as exc:
            if is_main_process(rank):
                print(f"[Warning] Scheduler state could not be restored ({exc}); continuing fresh scheduler state.")
        scaler.load_state_dict(ckpt["scaler"])
        start_epoch = int(ckpt.get("epoch", 0)) + 1
        global_step = int(ckpt.get("global_step", start_epoch * steps_per_epoch))
        best_psnr = float(ckpt.get("best_psnr", best_psnr))
        best_lpips = float(ckpt.get("best_lpips", best_lpips))
        if dc_sched is not None:
            dc_sched.load_state_dict(ckpt.get("dc_scheduler", {}))
        lpips_gate_epoch = ckpt.get("lpips_gate_epoch", lpips_gate_epoch)
        prev_val_psnr = ckpt.get("prev_val_psnr", prev_val_psnr)
        if is_main_process(rank):
            print(f"Resumed from {resume_path} at epoch {start_epoch}")

    if is_main_process(rank):
        with open(out_dir / "train_config.json", "w", encoding="utf-8") as f:
            json.dump(vars(args), f, indent=2)

    barrier(distributed)

    max_epochs = max(args.epochs, int(math.ceil(float(target_total_iters) / float(steps_per_epoch))))

    if is_main_process(rank):
        print(
            "Training schedule:",
            json.dumps(
                {
                    "target_total_iters": target_total_iters,
                    "steps_per_epoch": steps_per_epoch,
                    "warmup_iters": warmup_iters,
                    "max_epochs": max_epochs,
                    "finetune_preset": args.finetune_preset,
                    "stage1_end_epoch": args.stage1_end_epoch,
                    "stage2_end_epoch": args.stage2_end_epoch,
                    "augment_profile": args.augment_profile,
                    "augment_curriculum": args.augment_curriculum,
                    "ood_prob": args.ood_prob,
                    "ood_max_transforms": args.ood_max_transforms,
                    "pixel_loss_type": args.pixel_loss_type,
                    "optimizer_betas": [args.optim_beta1, args.optim_beta2],
                    "lr_eta_min": args.lr_eta_min,
                    "lambda_psnr": args.lambda_psnr,
                    "lambda_l1": args.lambda_l1,
                    "lambda_l2": args.lambda_l2,
                    "lambda_charbonnier": args.lambda_charbonnier,
                    "lambda_fft": args.lambda_fft,
                },
                indent=2,
            ),
        )

    freeze_phase_current: str | None = None
    freeze_trainable = 0
    freeze_total = 0

    for epoch in range(start_epoch, max_epochs):
        if global_step >= target_total_iters:
            break

        epoch_t0 = time.time()
        if train_sampler is not None:
            train_sampler.set_epoch(epoch)
        if val_sampler is not None:
            val_sampler.set_epoch(epoch)

        aug_policy = effective_augment_policy(args, epoch=epoch, max_epochs=max_epochs)
        train_ds.set_ood_policy(
            profile=str(aug_policy["profile"]),
            prob=float(aug_policy["ood_prob"]),
            max_transforms=int(aug_policy["ood_max_transforms"]),
        )

        freeze_phase = _freeze_phase_for_epoch(args, epoch)
        if freeze_phase != freeze_phase_current:
            freeze_trainable, freeze_total = _apply_freeze_phase(model, args, freeze_phase)
            freeze_phase_current = freeze_phase
            if is_main_process(rank):
                print(
                    f"[Freeze] phase={freeze_phase_current} "
                    f"trainable_params={freeze_trainable}/{freeze_total}"
                )

        model.train()
        running = {
            "loss_total": 0.0,
            "loss_recon": 0.0,
            "loss_psnr": 0.0,
            "loss_l1": 0.0,
            "loss_l2": 0.0,
            "loss_charbonnier": 0.0,
            "loss_fft": 0.0,
            "loss_ssim": 0.0,
            "loss_dc": 0.0,
            "loss_lpips": 0.0,
            "loss_edge": 0.0,
        }
        num_steps = 0
        skipped_nonfinite = 0

        iterator = tqdm(train_loader, disable=not is_main_process(rank), desc=f"Epoch {epoch+1}/{max_epochs}")

        h2_jitter_effective = effective_h2_jitter(args, epoch)
        stage_weights = effective_stage_weights(args, epoch=epoch, max_epochs=max_epochs)

        lam_psnr_effective = stage_weights["lambda_psnr"]
        lam_l1_effective = stage_weights["lambda_l1"]
        lam_l2_effective = stage_weights["lambda_l2"]
        lam_charb_effective = stage_weights["lambda_charbonnier"]
        lam_fft_effective = stage_weights["lambda_fft"]

        if args.finetune_preset == "none":
            lam_ssim_effective = effective_lambda_ssim(args, epoch)
            lam_dc_base = stage_weights["lambda_dc"]
        else:
            lam_ssim_effective = stage_weights["lambda_ssim"]
            lam_dc_base = stage_weights["lambda_dc"]

        if lpips_enabled and lpips_gate_epoch is None:
            gate_epoch_ready = epoch >= args.lpips_start_epoch and epoch >= args.lpips_gate_min_epoch
            gate_psnr_ready = prev_val_psnr is not None and prev_val_psnr >= args.lpips_gate_psnr
            if gate_epoch_ready and gate_psnr_ready:
                lpips_gate_epoch = epoch

        lam_lpips_effective = effective_lambda_lpips(args, epoch, lpips_gate_epoch) if lpips_enabled else 0.0

        if args.finetune_preset == "none":
            use_dc = lam_dc_base > 0 and epoch >= args.warmup_epochs
        else:
            use_dc = lam_dc_base > 0

        if use_dc:
            if dc_schedule_mode == "adaptive" and dc_sched is not None:
                lam_dc_effective = dc_sched.lam
            elif dc_schedule_mode == "constant" and args.finetune_preset == "none":
                lam_dc_effective = lam_dc_base
            else:
                if args.finetune_preset == "none":
                    dc_epoch = epoch - args.warmup_epochs
                    ramp_epochs = max(args.dc_ramp_epochs, 1)
                    lam_dc_effective = min(lam_dc_base, lam_dc_base * float(dc_epoch + 1) / float(ramp_epochs))
                else:
                    lam_dc_effective = lam_dc_base
        else:
            lam_dc_effective = 0.0

        for batch in iterator:
            if global_step >= target_total_iters:
                break

            y = batch["noisy"].to(device, non_blocking=True)
            x = batch["gt"].to(device, non_blocking=True)

            if h2_jitter_effective > 0:
                factor_a = 1.0 + random.uniform(-h2_jitter_effective, h2_jitter_effective)
                factor_b = 1.0 + random.uniform(-h2_jitter_effective, h2_jitter_effective)
                batch_loss_cfg = replace(
                    loss_cfg,
                    h2_a=max(loss_cfg.h2_a * factor_a, 1e-10),
                    h2_b=max(loss_cfg.h2_b * factor_b, 1e-10),
                    lambda_psnr=lam_psnr_effective,
                    lambda_l1=lam_l1_effective,
                    lambda_l2=lam_l2_effective,
                    lambda_charbonnier=lam_charb_effective,
                    lambda_fft=lam_fft_effective,
                    lambda_ssim=lam_ssim_effective,
                    lambda_dc=lam_dc_effective,
                    lambda_lpips=lam_lpips_effective,
                )
            else:
                batch_loss_cfg = replace(
                    loss_cfg,
                    lambda_psnr=lam_psnr_effective,
                    lambda_l1=lam_l1_effective,
                    lambda_l2=lam_l2_effective,
                    lambda_charbonnier=lam_charb_effective,
                    lambda_fft=lam_fft_effective,
                    lambda_ssim=lam_ssim_effective,
                    lambda_dc=lam_dc_effective,
                    lambda_lpips=lam_lpips_effective,
                )

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type=device.type, enabled=(args.amp and device.type == "cuda")):
                pred = model(y)

                if not _all_ranks_finite(pred, distributed=distributed):
                    skipped_nonfinite += 1
                    optimizer.zero_grad(set_to_none=True)
                    if is_main_process(rank) and skipped_nonfinite <= 3:
                        print(
                            f"[Warning] Non-finite prediction detected at epoch={epoch + 1}, "
                            f"step={num_steps + 1}. Skipping batch."
                        )
                    continue

                if (
                    use_dc
                    and args.dc_debug_first_epoch
                    and epoch == args.warmup_epochs
                    and num_steps == 0
                    and is_main_process(rank)
                ):
                    with torch.no_grad():
                        mu_dbg, q_dbg = forward_consistency_h2(pred, scale=batch_loss_cfg.scale)
                        v_dbg = heteroscedastic_variance_h2(q_dbg, h2_a=batch_loss_cfg.h2_a, h2_b=batch_loss_cfg.h2_b)
                        resid_abs = (y - mu_dbg).abs()
                        print(
                            "[DC debug] "
                            f"epoch={epoch + 1} "
                            f"lam_dc={lam_dc_effective:.5f} "
                            f"v[min,max,mean]=({v_dbg.min().item():.5f},{v_dbg.max().item():.5f},{v_dbg.mean().item():.5f}) "
                            f"|resid|[min,max,mean]=({resid_abs.min().item():.5f},{resid_abs.max().item():.5f},{resid_abs.mean().item():.5f})"
                        )

                loss, logs = combined_restoration_loss(
                    x_hat=pred,
                    x_gt=x,
                    y_lr=y,
                    cfg=batch_loss_cfg,
                    use_dc=use_dc,
                    lpips_net=lpips_net,
                )

            if not _all_ranks_finite(loss, distributed=distributed):
                skipped_nonfinite += 1
                optimizer.zero_grad(set_to_none=True)
                if is_main_process(rank) and skipped_nonfinite <= 3:
                    print(
                        f"[Warning] Non-finite loss detected at epoch={epoch + 1}, "
                        f"step={num_steps + 1}. Skipping batch."
                    )
                continue

            scaler.scale(loss).backward()
            if args.grad_clip > 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            global_step += 1
            if global_step > 1:
                scheduler.step()

            num_steps += 1
            for k in running:
                running[k] += logs[k]

            if is_main_process(rank):
                iterator.set_postfix(
                    loss=f"{logs['loss_total']:.4f}",
                    recon=f"{logs['loss_recon']:.4f}",
                    psnr=f"{logs['loss_psnr']:.4f}",
                    l1=f"{logs['loss_l1']:.4f}",
                    l2=f"{logs['loss_l2']:.4f}",
                    charb=f"{logs['loss_charbonnier']:.4f}",
                    fft=f"{logs['loss_fft']:.4f}",
                    dc=f"{logs['loss_dc']:.4f}",
                    ssim=f"{logs['loss_ssim']:.4f}",
                    edge=f"{logs['loss_edge']:.4f}",
                    lpips=f"{logs['loss_lpips']:.4f}",
                    lam_psnr=f"{lam_psnr_effective:.3f}",
                    lam_l1=f"{lam_l1_effective:.3f}",
                    lam_l2=f"{lam_l2_effective:.3f}",
                    lam_charb=f"{lam_charb_effective:.3f}",
                    lam_fft=f"{lam_fft_effective:.3f}",
                    lam_dc=f"{lam_dc_effective:.4f}",
                    lam_ssim=f"{lam_ssim_effective:.4f}",
                    lam_lpips=f"{lam_lpips_effective:.4f}",
                    h2_jit=f"{h2_jitter_effective:.4f}",
                    dc_on=str(use_dc),
                    it=f"{global_step}/{target_total_iters}",
                )

        if num_steps == 0:
            if global_step >= target_total_iters:
                break
            raise RuntimeError(
                "All batches in this epoch were skipped due to non-finite tensors. "
                "Try lowering LR and disabling AMP (amp=false)."
            )

        epoch_metrics = {k: running[k] / max(num_steps, 1) for k in running}
        epoch_metrics = reduce_dict(epoch_metrics, device=device, distributed=distributed)

        val_metrics = evaluate(model, val_loader, device=device, distributed=distributed, lpips_net=lpips_net)
        prev_val_psnr = float(val_metrics.get("val_psnr", 0.0))

        lam_dc_next = lam_dc_effective
        if use_dc and dc_schedule_mode == "adaptive" and dc_sched is not None and val_metrics:
            lam_dc_next = dc_sched.update(float(val_metrics.get("val_psnr", 0.0)))

        if is_main_process(rank):
            elapsed = time.time() - epoch_t0
            msg = {
                "epoch": epoch,
                "global_step": global_step,
                "target_total_iters": target_total_iters,
                "elapsed_sec": round(elapsed, 2),
                "train": epoch_metrics,
                "val": val_metrics,
                "use_dc": use_dc,
                "dc_schedule": dc_schedule_mode,
                "finetune_preset": args.finetune_preset,
                "lam_psnr": round(float(lam_psnr_effective), 6),
                "lam_l1": round(float(lam_l1_effective), 6),
                "lam_l2": round(float(lam_l2_effective), 6),
                "lam_charbonnier": round(float(lam_charb_effective), 6),
                "lam_fft": round(float(lam_fft_effective), 6),
                "lam_dc": round(float(lam_dc_effective), 6),
                "lam_dc_next": round(float(lam_dc_next), 6),
                "lam_ssim": round(float(lam_ssim_effective), 6),
                "lam_lpips": round(float(lam_lpips_effective), 6),
                "augment_profile": str(aug_policy["profile"]),
                "ood_prob": round(float(aug_policy["ood_prob"]), 4),
                "ood_max_transforms": int(aug_policy["ood_max_transforms"]),
                "lpips_gate_epoch": lpips_gate_epoch,
                "freeze_phase": freeze_phase_current,
                "trainable_params": int(freeze_trainable),
                "total_params": int(freeze_total),
                "skipped_nonfinite": int(skipped_nonfinite),
                "h2_jitter": round(float(h2_jitter_effective), 6),
            }
            print(json.dumps(msg))

            out_dir.mkdir(parents=True, exist_ok=True)

            save_payload = {
                "epoch": epoch,
                "model": (model.module if isinstance(model, DDP) else model).state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict(),
                "global_step": global_step,
                "target_total_iters": target_total_iters,
                "best_psnr": best_psnr,
                "best_lpips": best_lpips,
                "dc_schedule_mode": dc_schedule_mode,
                "dc_scheduler": dc_sched.state_dict() if dc_sched is not None else None,
                "lpips_gate_epoch": lpips_gate_epoch,
                "prev_val_psnr": prev_val_psnr,
                "pretrained_report": pretrained_report,
                "args": vars(args),
            }

            if args.save_every > 0 and (epoch + 1) % args.save_every == 0:
                torch.save(save_payload, out_dir / f"checkpoint_epoch_{epoch+1:03d}.pt")

            cur_psnr = float(val_metrics["val_psnr"])
            if cur_psnr > best_psnr:
                best_psnr = cur_psnr
                save_payload["best_psnr"] = best_psnr
                torch.save(save_payload, out_dir / "best.pt")
                if args.export_best_infer:
                    torch.save(build_infer_export_payload(save_payload), out_dir / "best_infer.pt")

            cur_lpips = float(val_metrics.get("val_lpips", float("inf")))
            if lpips_enabled and np.isfinite(cur_lpips) and cur_lpips < best_lpips:
                best_lpips = cur_lpips
                save_payload["best_lpips"] = best_lpips
                torch.save(save_payload, out_dir / "best_lpips.pt")
                if args.export_best_infer:
                    torch.save(build_infer_export_payload(save_payload), out_dir / "best_lpips_infer.pt")

            torch.save(save_payload, out_dir / "latest.pt")
            if args.export_latest_infer:
                torch.save(build_infer_export_payload(save_payload), out_dir / "latest_infer.pt")

    if is_main_process(rank) and args.export_best_infer:
        best_infer = out_dir / "best_infer.pt"
        latest_infer = out_dir / "latest_infer.pt"
        if not best_infer.exists() and latest_infer.exists():
            shutil.copyfile(latest_infer, best_infer)

    barrier(distributed)
    cleanup_distributed(distributed)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/infer_nafnet_ddp.py
from __future__ import annotations

import argparse
import base64
import csv
import json
import os
from io import BytesIO
from pathlib import Path
from typing import Sequence

import numpy as np
import torch
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler
from tqdm import tqdm

from src.model import NAFNetH2SR, make_nafnet_config
from src.config import apply_config_defaults, load_config_file
from src.dataset import NpyNoisyDataset
from src.ddp import (
    barrier,
    cleanup_distributed,
    cuda_environment_summary,
    init_distributed_mode,
    is_main_process,
)
from src.losses import dc_loss_robust, forward_consistency_h2, heteroscedastic_variance_h2


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description="Distributed inference for NAFNet H2 model")
    parser.add_argument("--config", type=str, default="", help="Path to YAML/JSON config file")

    parser.add_argument("--input-dir", type=str, default="../data/test/NoisyLR")
    parser.add_argument("--output-dir", type=str, default="outputs/nafnet_h2/test_predictions")
    parser.add_argument("--checkpoint", type=str, default="auto")
    parser.add_argument("--train-output-dir", type=str, default="outputs/nafnet_h2")

    parser.add_argument("--batch-size", type=int, default=8)
    parser.add_argument("--num-workers", type=int, default=4)
    parser.add_argument("--amp", action=argparse.BooleanOptionalAction, default=True)
    parser.add_argument("--quality-preset", type=str, default="fast", choices=["fast", "balanced", "high", "custom"])

    parser.add_argument("--nafnet-preset", type=str, default=None)
    parser.add_argument("--official-repo", type=str, default=None)
    parser.add_argument("--in-channels", type=int, default=None)
    parser.add_argument("--scale", type=int, default=None)
    parser.add_argument("--upsample-mode", type=str, default=None)

    parser.add_argument("--h2-a", type=float, default=0.14160734)
    parser.add_argument("--h2-b", type=float, default=6.3383e-05)
    parser.add_argument("--student-t-nu", type=float, default=3.0)

    parser.add_argument("--refine-steps", type=int, default=None)
    parser.add_argument("--refine-lr", type=float, default=None)
    parser.add_argument("--tta", type=str, default=None, choices=["none", "flip4", "flip8"])

    parser.add_argument("--write-submission-csv", action=argparse.BooleanOptionalAction, default=False)
    parser.add_argument("--submission-csv-path", type=str, default="")
    parser.add_argument("--verify-all-inputs", action=argparse.BooleanOptionalAction, default=True)
    parser.add_argument("--expected-size", type=int, default=256)
    parser.add_argument("--expected-count", type=int, default=-1)

    return parser


def parse_args() -> argparse.Namespace:
    pre = argparse.ArgumentParser(add_help=False)
    pre.add_argument("--config", type=str, default="")
    known, _ = pre.parse_known_args()

    parser = build_parser()
    if known.config:
        cfg_data = load_config_file(known.config)
        unknown = apply_config_defaults(parser, cfg_data)
        if unknown:
            print(f"[Config] Ignored unknown keys: {sorted(list(unknown.keys()))}")

    return parser.parse_args()


def resolve_inference_quality(args: argparse.Namespace) -> tuple[int, float, str]:
    preset_defaults = {
        "fast": {"refine_steps": 0, "refine_lr": 1e-3, "tta": "none"},
        "balanced": {"refine_steps": 3, "refine_lr": 8e-4, "tta": "flip4"},
        "high": {"refine_steps": 8, "refine_lr": 5e-4, "tta": "flip8"},
        "custom": {"refine_steps": 0, "refine_lr": 1e-3, "tta": "none"},
    }

    base = preset_defaults[args.quality_preset]
    refine_steps = int(args.refine_steps) if args.refine_steps is not None else int(base["refine_steps"])
    refine_lr = float(args.refine_lr) if args.refine_lr is not None else float(base["refine_lr"])
    tta_mode = str(args.tta) if args.tta is not None else str(base["tta"])

    if refine_steps < 0:
        raise ValueError("refine_steps must be >= 0")
    if refine_steps > 0 and refine_lr <= 0:
        raise ValueError("refine_lr must be > 0 when refine_steps > 0")

    return refine_steps, refine_lr, tta_mode


def resolve_checkpoint_path(checkpoint_arg: str, train_output_dir: str) -> Path:
    if checkpoint_arg and checkpoint_arg.lower() != "auto":
        p = Path(checkpoint_arg)
        if not p.exists():
            raise FileNotFoundError(f"Checkpoint not found: {p}")
        return p

    base = Path(train_output_dir)
    candidates = [
        base / "best_lpips_infer.pt",
        base / "best_infer.pt",
        base / "best_lpips.pt",
        base / "best.pt",
        base / "latest_infer.pt",
        base / "latest.pt",
    ]
    for p in candidates:
        if p.exists():
            return p

    raise FileNotFoundError(
        f"Could not auto-resolve checkpoint in {base}. Expected one of: "
        "best_lpips_infer.pt, best_infer.pt, best_lpips.pt, best.pt, latest_infer.pt, latest.pt"
    )


def validate_prediction_array(arr: np.ndarray, expected_size: int) -> np.ndarray:
    if arr.ndim == 3 and arr.shape[0] == 1:
        arr = arr[0]
    if arr.ndim != 2:
        raise ValueError(f"Expected 2D array, got shape {arr.shape}")
    if arr.shape != (expected_size, expected_size):
        raise ValueError(f"Expected shape ({expected_size}, {expected_size}), got {arr.shape}")

    arr = arr.astype(np.float32, copy=False)
    if not np.all(np.isfinite(arr)):
        raise ValueError("Prediction contains NaN or Inf values")
    return arr


def write_submission_csv(submission_dir: Path, csv_path: Path, expected_count: int, expected_size: int) -> int:
    files = sorted([f for f in os.listdir(submission_dir) if f.endswith(".npy")])
    if expected_count >= 0 and len(files) != expected_count:
        raise RuntimeError(f"Expected {expected_count} .npy files in {submission_dir}, found {len(files)}")

    rows = []
    for idx, file_name in enumerate(files, start=1):
        arr = np.load(submission_dir / file_name)
        arr = validate_prediction_array(arr, expected_size=expected_size)

        buffer = BytesIO()
        np.save(buffer, arr)
        encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")
        rows.append({"id": idx, "npy_base64": encoded})

    csv_path.parent.mkdir(parents=True, exist_ok=True)
    with csv_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["id", "npy_base64"])
        writer.writeheader()
        writer.writerows(rows)

    return len(rows)


def build_model_from_checkpoint_args(args: argparse.Namespace, ckpt_args: dict, device: torch.device) -> torch.nn.Module:
    nafnet_preset = args.nafnet_preset if args.nafnet_preset is not None else ckpt_args.get("nafnet_preset", "sidd-width64")
    official_repo = args.official_repo if args.official_repo is not None else ckpt_args.get("official_repo", "nafnet/official")
    in_channels = args.in_channels if args.in_channels is not None else ckpt_args.get("in_channels", 1)
    scale = args.scale if args.scale is not None else ckpt_args.get("scale", 2)
    upsample_mode = args.upsample_mode if args.upsample_mode is not None else ckpt_args.get("upsample_mode", "bilinear")

    cfg = make_nafnet_config(
        preset=nafnet_preset,
        in_channels=in_channels,
        out_channels=in_channels,
        scale=scale,
        official_repo=official_repo,
        upsample_mode=upsample_mode,
    )
    model = NAFNetH2SR(cfg).to(device)
    return model


def _apply_ops(x: torch.Tensor, ops: Sequence[str]) -> torch.Tensor:
    out = x
    for op in ops:
        if op == "h":
            out = torch.flip(out, dims=(-1,))
        elif op == "v":
            out = torch.flip(out, dims=(-2,))
        elif op == "t":
            out = out.transpose(-2, -1)
        else:
            raise ValueError(f"Unknown TTA op: {op}")
    return out


def _tta_opsets(mode: str) -> list[tuple[str, ...]]:
    if mode == "none":
        return [tuple()]
    if mode == "flip4":
        return [tuple(), ("h",), ("v",), ("h", "v")]
    if mode == "flip8":
        return [tuple(), ("h",), ("v",), ("h", "v"), ("t",), ("t", "h"), ("t", "v"), ("t", "h", "v")]
    raise ValueError(f"Unknown tta mode: {mode}")


def predict_with_tta(model: torch.nn.Module, y: torch.Tensor, tta_mode: str) -> torch.Tensor:
    opsets = _tta_opsets(tta_mode)
    pred_sum = None
    for ops in opsets:
        y_aug = _apply_ops(y, ops)
        pred_aug = model(y_aug)
        pred = _apply_ops(pred_aug, tuple(reversed(ops)))
        if pred_sum is None:
            pred_sum = pred
        else:
            pred_sum = pred_sum + pred

    if pred_sum is None:
        raise RuntimeError("TTA prediction failed: no augmentations were applied")
    return pred_sum / float(len(opsets))


def refine_with_data_consistency(
    pred_hr: torch.Tensor,
    y_lr: torch.Tensor,
    h2_a: float,
    h2_b: float,
    nu: float,
    scale: int,
    steps: int,
    lr: float,
) -> torch.Tensor:
    if steps <= 0:
        return pred_hr

    x_opt = pred_hr.detach().clone().requires_grad_(True)
    opt = torch.optim.Adam([x_opt], lr=lr)

    for _ in range(steps):
        mu, q = forward_consistency_h2(x_opt, scale=scale)
        var = heteroscedastic_variance_h2(q, h2_a=h2_a, h2_b=h2_b)
        loss = dc_loss_robust(y_lr, mu, var, nu=nu)

        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()

        with torch.no_grad():
            x_opt.clamp_(0.0, 1.0)

    return x_opt.detach()


def main() -> None:
    args = parse_args()
    refine_steps, refine_lr, tta_mode = resolve_inference_quality(args)

    env = init_distributed_mode()
    rank = env["rank"]
    world_size = env["world_size"]
    distributed = env["distributed"]
    device = env["device"]

    if is_main_process(rank):
        print("CUDA environment:", json.dumps(cuda_environment_summary(), indent=2))
        print(f"Distributed: {distributed}, world_size={world_size}")
        print(
            "Inference quality:",
            json.dumps(
                {
                    "quality_preset": args.quality_preset,
                    "refine_steps": refine_steps,
                    "refine_lr": refine_lr,
                    "tta": tta_mode,
                },
                indent=2,
            ),
        )

    output_dir = Path(args.output_dir)
    if is_main_process(rank):
        output_dir.mkdir(parents=True, exist_ok=True)
    barrier(distributed)

    checkpoint_path = resolve_checkpoint_path(args.checkpoint, args.train_output_dir)
    if is_main_process(rank):
        print(f"Using checkpoint: {checkpoint_path}")

    ckpt = torch.load(checkpoint_path, map_location="cpu")
    ckpt_args = ckpt.get("args", {})

    model = build_model_from_checkpoint_args(args, ckpt_args=ckpt_args, device=device)
    model.load_state_dict(ckpt["model"], strict=True)
    model.eval()

    scale = int(args.scale if args.scale is not None else ckpt_args.get("scale", 2))

    if distributed:
        model = DDP(model, device_ids=[env["local_rank"]] if device.type == "cuda" else None)

    dataset = NpyNoisyDataset(args.input_dir)
    sampler = None
    if distributed:
        sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=False, drop_last=False)

    loader = DataLoader(
        dataset,
        batch_size=args.batch_size,
        shuffle=False,
        sampler=sampler,
        num_workers=args.num_workers,
        pin_memory=(device.type == "cuda"),
        drop_last=False,
        persistent_workers=(args.num_workers > 0),
    )

    iterator = tqdm(loader, disable=not is_main_process(rank), desc="Inference")

    for batch in iterator:
        y = batch["noisy"].to(device, non_blocking=True)
        names = batch["name"]

        with torch.no_grad():
            with torch.amp.autocast(device_type=device.type, enabled=(args.amp and device.type == "cuda")):
                pred = predict_with_tta(model, y, tta_mode=tta_mode)

        if refine_steps > 0:
            pred = refine_with_data_consistency(
                pred_hr=pred,
                y_lr=y,
                h2_a=args.h2_a,
                h2_b=args.h2_b,
                nu=args.student_t_nu,
                scale=scale,
                steps=refine_steps,
                lr=refine_lr,
            )

        pred = pred.clamp(0.0, 1.0)

        for i, name in enumerate(names):
            arr = pred[i].detach().cpu().numpy()
            arr = validate_prediction_array(arr, expected_size=args.expected_size)
            np.save(output_dir / name, arr)

    barrier(distributed)
    if is_main_process(rank):
        produced = len([f for f in os.listdir(output_dir) if f.endswith(".npy")])
        if args.verify_all_inputs and produced != len(dataset):
            raise RuntimeError(f"Expected {len(dataset)} prediction files, found {produced}")

        print(f"Saved {produced} predictions to {output_dir}")

        if args.write_submission_csv:
            csv_path = Path(args.submission_csv_path) if args.submission_csv_path else output_dir.parent / "submission.csv"
            expected_count = args.expected_count if args.expected_count >= 0 else len(dataset)
            rows = write_submission_csv(
                submission_dir=output_dir,
                csv_path=csv_path,
                expected_count=expected_count,
                expected_size=args.expected_size,
            )
            print(f"Wrote submission CSV with {rows} rows to {csv_path}")

    cleanup_distributed(distributed)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/make_submission_csv.py
from __future__ import annotations

import argparse
import base64
import csv
import os
from io import BytesIO
from pathlib import Path

import numpy as np


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Create submission.csv from .npy prediction files")
    parser.add_argument("--submission-dir", type=str, default="/kaggle/working/submission")
    parser.add_argument("--output-csv", type=str, default="/kaggle/working/submission.csv")
    parser.add_argument("--expected-size", type=int, default=256)
    parser.add_argument("--expected-count", type=int, default=-1)
    return parser.parse_args()


def validate_array(arr: np.ndarray, expected_size: int) -> np.ndarray:
    if arr.ndim != 2:
        raise ValueError(f"Expected 2D array, got shape {arr.shape}")
    if arr.shape != (expected_size, expected_size):
        raise ValueError(f"Expected shape ({expected_size}, {expected_size}), got {arr.shape}")

    arr = arr.astype(np.float32, copy=False)
    if not np.all(np.isfinite(arr)):
        raise ValueError("Array contains NaN or Inf values")
    return arr


def main() -> None:
    args = parse_args()

    submission_dir = Path(args.submission_dir)
    output_csv = Path(args.output_csv)
    output_csv.parent.mkdir(parents=True, exist_ok=True)

    files = sorted([f for f in os.listdir(submission_dir) if f.endswith(".npy")])
    if args.expected_count >= 0 and len(files) != args.expected_count:
        raise RuntimeError(
            f"Expected {args.expected_count} .npy files, found {len(files)} in {submission_dir}"
        )

    rows = []
    for idx, file_name in enumerate(files, start=1):
        path = submission_dir / file_name
        arr = np.load(path)
        arr = validate_array(arr, expected_size=args.expected_size)

        buffer = BytesIO()
        np.save(buffer, arr)
        encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")
        rows.append({"id": idx, "npy_base64": encoded})

    with output_csv.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["id", "npy_base64"])
        writer.writeheader()
        writer.writerows(rows)

    print(f"Submission created with {len(rows)} rows at {output_csv}")


if __name__ == "__main__":
    main()


In [ ]:
import os
written = [f for f in os.listdir("src") if f.endswith(".py")]
print("src/ contents:", sorted(written))
assert len(written) >= 8, "Some source files are missing!"
print("All source files present ✓")


## ⚙️ Configuration
Edit the variables below to match your Kaggle dataset.


In [ ]:
import json, os

# ================================================================
#  ← EDIT THESE PATHS ←
# ================================================================
COMPETITION       = "my-competition"            # Kaggle dataset slug
DATA_ROOT         = f"/kaggle/input/kla-hack-x/train"
GT_DIR            = f"{DATA_ROOT}/GT"
NOISY_DIR         = f"{DATA_ROOT}/NoisyLR"
TEST_DIR          = f"/kaggle/input/kla-hack-x/test/NoisyLR"

OUTPUT_DIR        = "outputs/nafnet_h2"          # training checkpoints
SUBMISSION_DIR    = "/kaggle/working/submission"  # per-sample .npy predictions
SUBMISSION_CSV    = "/kaggle/working/submission.csv"

# Pretrained SIDD weights (see Cell 15 to auto-download)
PRETRAINED        = "nafnet/weights/nafnet_sidd_width64.pth"

# ================================================================
#  TRAINING HYPERPARAMETERS
# ================================================================
EPOCHS            = 120
BATCH_SIZE        = 8
PATCH_SIZE        = 256
LR                = 5e-4
WEIGHT_DECAY      = 0.0
GRAD_CLIP         = 0.0
AMP               = True       # False to disable mixed precision (fixes some NaN runs)
NUM_WORKERS       = 4
VAL_RATIO         = 0.10
SEED              = 42

# Model
NAFNET_PRESET     = "sidd-width64"   # "sidd-width32" for lighter model
IN_CHANNELS       = 1
SCALE             = 2
UPSAMPLE_MODE     = "bilinear"

# Loss
FINETUNE_PRESET   = "roi_psnr_120e"  # "none" for manual lambda control
LAMBDA_PSNR       = 1.0
LAMBDA_SSIM       = 0.02
LAMBDA_DC         = 0.008

# Augmentation curriculum
AUGMENT_CURRICULUM = True
OOD_PROB          = 0.95
OOD_MAX_TRANSFORMS = 5

# LR schedule
LR_WARMUP_EPOCHS  = 3
LR_ETA_MIN        = 1e-7

# Inference quality: "fast" | "balanced" | "high"
QUALITY_PRESET    = "fast"
EXPECTED_SIZE     = 256  # output image side length in pixels

# ================================================================
#  WRITE CONFIG FILES  (used by the training / inference scripts)
# ================================================================
os.makedirs(OUTPUT_DIR, exist_ok=True)

train_cfg = dict(
    data_root          = DATA_ROOT,
    gt_subdir          = "GT",
    noisy_subdir       = "NoisyLR",
    output_dir         = OUTPUT_DIR,
    nafnet_preset      = NAFNET_PRESET,
    official_repo      = "nafnet/official",
    pretrained         = PRETRAINED,
    strict_pretrained  = False,
    staged_freeze      = False,
    freeze_stage1_end_epoch = 10,
    freeze_stage2_end_epoch = 25,
    freeze_intro       = True,
    in_channels        = IN_CHANNELS,
    scale              = SCALE,
    upsample_mode      = UPSAMPLE_MODE,
    epochs             = EPOCHS,
    batch_size         = BATCH_SIZE,
    num_workers        = NUM_WORKERS,
    lr                 = LR,
    weight_decay       = WEIGHT_DECAY,
    grad_clip          = GRAD_CLIP,
    optim_beta1        = 0.9,
    optim_beta2        = 0.9,
    amp                = AMP,
    lr_warmup_epochs   = LR_WARMUP_EPOCHS,
    lr_warmup_start_factor = 0.05,
    lr_eta_min         = LR_ETA_MIN,
    total_iters        = -1,
    patch_size         = PATCH_SIZE,
    augment_profile    = "max",
    ood_prob           = OOD_PROB,
    ood_max_transforms = OOD_MAX_TRANSFORMS,
    ood_clip_min       = -0.25,
    ood_clip_max       = 1.8,
    augment_curriculum = AUGMENT_CURRICULUM,
    val_ratio          = VAL_RATIO,
    seed               = SEED,
    finetune_preset    = FINETUNE_PRESET,
    stage1_end_epoch   = 20,
    stage2_end_epoch   = 90,
    lambda_psnr        = LAMBDA_PSNR,
    lambda_l1          = 0.0,
    lambda_l2          = 0.0,
    lambda_charbonnier = 0.0,
    lambda_fft         = 0.0,
    warmup_epochs      = 0,
    dc_schedule        = "constant",
    dc_lambda_start    = 0.005,
    dc_lambda_step     = 0.005,
    dc_lambda_cap      = 0.04,
    dc_patience        = 2,
    dc_min_delta       = 0.0001,
    dc_ramp_epochs     = 10,
    lambda_ssim        = LAMBDA_SSIM,
    ssim_start_epoch   = 0,
    ssim_lambda_start  = 0.0,
    ssim_ramp_epochs   = 1,
    ssim_contribute_to_loss = True,
    pixel_loss_type    = "psnr",
    charbonnier_eps    = 0.001,
    lambda_edge        = 0.0,
    lambda_dc          = LAMBDA_DC,
    student_t_nu       = 3.0,
    lambda_lpips_max   = 0.0,
    lpips_start_epoch  = 60,
    lpips_lambda_start = 0.0,
    lpips_ramp_epochs  = 30,
    lpips_gate_psnr    = 25.5,
    lpips_gate_min_epoch = 40,
    lpips_net          = "alex",
    require_ssim       = True,
    require_lpips      = False,
    dc_debug_first_epoch = False,
    h2_a               = 0.14160734,
    h2_b               = 6.3383e-05,
    h2_jitter          = 0.0,
    h2_jitter_decay    = "none",
    h2_jitter_min      = 0.0,
    resume             = "",
    auto_resume        = True,
    save_every         = -1,
    export_best_infer  = True,
    export_latest_infer = True,
)

with open("train_config.json", "w") as f:
    json.dump(train_cfg, f, indent=2)

infer_cfg = dict(
    input_dir        = TEST_DIR,
    output_dir       = SUBMISSION_DIR,
    train_output_dir = OUTPUT_DIR,
    checkpoint       = "auto",
    batch_size       = 8,
    num_workers      = NUM_WORKERS,
    amp              = AMP,
    quality_preset   = QUALITY_PRESET,
    nafnet_preset    = NAFNET_PRESET,
    official_repo    = "nafnet/official",
    in_channels      = IN_CHANNELS,
    scale            = SCALE,
    upsample_mode    = UPSAMPLE_MODE,
    h2_a             = 0.14160734,
    h2_b             = 6.3383e-05,
    student_t_nu     = 3.0,
    verify_all_inputs = False,
    expected_size    = EXPECTED_SIZE,
    write_submission_csv = False,
)

with open("infer_config.json", "w") as f:
    json.dump(infer_cfg, f, indent=2)

print("Config files written: train_config.json, infer_config.json")
print(f"\n  GT_DIR    = {GT_DIR}")
print(f"  NOISY_DIR = {NOISY_DIR}")
print(f"  TEST_DIR  = {TEST_DIR}")
print(f"  PRETRAINED= {PRETRAINED}")
print(f"  EPOCHS    = {EPOCHS}  BATCH = {BATCH_SIZE}  LR = {LR}")


In [ ]:
import torch, json

print("PyTorch:", torch.__version__)
cuda = torch.cuda.is_available()
n = torch.cuda.device_count()
print(f"CUDA available: {cuda}   GPUs: {n}")
for i in range(n):
    mem = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"  [{i}] {torch.cuda.get_device_name(i)}  {mem:.1f} GB")

if n > 1:
    print(f"\n✓ Multi-GPU detected — training will use torchrun --nproc_per_node={n}")
elif n == 1:
    print("\n✓ Single GPU — training will use plain python")
else:
    print("\n⚠ No GPU — training on CPU (very slow!)")


In [ ]:
import os, numpy as np
from pathlib import Path

for label, d in [("GT", GT_DIR), ("NoisyLR", NOISY_DIR), ("Test NoisyLR", TEST_DIR)]:
    p = Path(d)
    if p.exists():
        files = sorted(p.glob("*.npy"))
        print(f"  {label:15s} : {len(files):5d} files  [{p}]")
        if files:
            sample = np.load(str(files[0]))
            print(f"  {'':15s}   sample shape={sample.shape}  dtype={sample.dtype}"
                  f"  range=[{sample.min():.3f}, {sample.max():.3f}]")
    else:
        print(f"  {label:15s} : ✗ NOT FOUND [{d}]")


## 📦 Pretrained Weights (optional but strongly recommended)

The cell below tries to download the official **NAFNet-SIDD-width64** checkpoint from the  
NAFNet GitHub releases. Skip or replace the URL if you have weights from a Kaggle dataset.

| Model | Params | URL |
|-------|--------|-----|
| NAFNet-SIDD-width64 | ~67M | https://github.com/megvii-research/NAFNet/releases/download/v0.1/NAFNet-SIDD-width64.pth |
| NAFNet-SIDD-width32 | ~17M | https://github.com/megvii-research/NAFNet/releases/download/v0.1/NAFNet-SIDD-width32.pth |


In [ ]:
import os, subprocess

NAFNET_WEIGHTS_URL = (
    "https://github.com/megvii-research/NAFNet/releases/download/v0.1/"
    "NAFNet-SIDD-width64.pth"
)

dst = PRETRAINED  # set in Configuration cell

if os.path.exists(dst):
    size_mb = os.path.getsize(dst) / 1e6
    print(f"Weights already present: {dst}  ({size_mb:.1f} MB) — skipping download.")
else:
    print(f"Downloading to {dst} ...")
    r = subprocess.run(["wget", "-q", "-O", dst, NAFNET_WEIGHTS_URL],
                       capture_output=True, text=True)
    if r.returncode == 0:
        size_mb = os.path.getsize(dst) / 1e6
        print(f"Downloaded: {dst}  ({size_mb:.1f} MB)")
    else:
        print(f"wget failed (may be blocked on Kaggle):\n{r.stderr}")
        print("\nAlternatives:")
        print("  1. Add NAFNet weights as a Kaggle dataset and update PRETRAINED path.")
        print("  2. Leave PRETRAINED='' to train from scratch (slower convergence).")


## 🏋️ Training

The cell below launches `src/train_nafnet_ddp.py` — the exact production training script —  
using `torchrun` (multi-GPU) or plain Python (single GPU).  

**Auto-resume** is enabled by default: re-run this cell to continue from the latest checkpoint.


In [ ]:
import subprocess, sys, os, torch

n_gpus = torch.cuda.device_count()

if n_gpus > 1:
    cmd = [
        "torchrun",
        f"--nproc_per_node={n_gpus}",
        "--master_port=29500",
        "src/train_nafnet_ddp.py",
        "--config", "train_config.json",
    ]
    print(f"Launching with torchrun on {n_gpus} GPUs …")
else:
    cmd = [sys.executable, "src/train_nafnet_ddp.py", "--config", "train_config.json"]
    print(f"Launching on {'1 GPU' if n_gpus == 1 else 'CPU'} …")

print(f"Command: {' '.join(cmd)}\n{'='*70}")

env = os.environ.copy()
env["PYTHONPATH"] = "." + os.pathsep + "nafnet/official" + os.pathsep + env.get("PYTHONPATH", "")

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)
try:
    for line in process.stdout:
        print(line, end="", flush=True)
finally:
    process.wait()

print(f"\n{'='*70}")
print(f"Training exited with code: {process.returncode}")
if process.returncode != 0:
    print("⚠ Non-zero exit — check output above for errors.")
else:
    print("✓ Training complete.")

# Show best checkpoint info
import json, pathlib
best_files = ["best_infer.pt", "best.pt", "latest_infer.pt", "latest.pt"]
for name in best_files:
    p = pathlib.Path(OUTPUT_DIR) / name
    if p.exists():
        ckpt = torch.load(str(p), map_location="cpu")
        psnr = ckpt.get("best_psnr", "n/a")
        epoch = ckpt.get("epoch", "n/a")
        print(f"\nCheckpoint: {p.name}  |  epoch={epoch}  best_psnr={psnr}")
        break


## 📊 Training Curves

In [ ]:
import json, os
from pathlib import Path

hist_path = Path(OUTPUT_DIR) / "history.json"  # written by newer runs; may not exist
# Fall back to parsing the JSONL epoch logs printed during training
if not hist_path.exists():
    print(f"No history.json found at {hist_path}. Re-run after training completes.")
else:
    with open(hist_path) as f:
        history = json.load(f)

    try:
        import matplotlib.pyplot as plt

        epochs   = [h["epoch"] for h in history]
        t_psnr   = [h.get("train", {}).get("loss_psnr", None) for h in history]
        v_psnr   = [h.get("val",   {}).get("val_psnr",  None) for h in history]
        v_ssim   = [h.get("val",   {}).get("val_ssim",  None) for h in history]

        fig, axes = plt.subplots(1, 2, figsize=(13, 4))

        ax = axes[0]
        if any(x is not None for x in t_psnr):
            ax.plot(epochs, t_psnr, label="train PSNR-loss", alpha=0.7)
        if any(x is not None for x in v_psnr):
            ax.plot(epochs, v_psnr, label="val PSNR (dB)", linewidth=2)
        ax.set_xlabel("Epoch"); ax.set_ylabel("PSNR"); ax.legend(); ax.grid(True, alpha=0.3)
        ax.set_title("PSNR over Training")

        ax = axes[1]
        if any(x is not None for x in v_ssim):
            ax.plot(epochs, v_ssim, color="orange", linewidth=2)
        ax.set_xlabel("Epoch"); ax.set_ylabel("SSIM"); ax.grid(True, alpha=0.3)
        ax.set_title("Validation SSIM over Training")

        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, "training_curves.png"), dpi=100)
        plt.show()

        best_epoch = max(range(len(history)), key=lambda i: history[i].get("val", {}).get("val_psnr", 0))
        best = history[best_epoch]
        print(f"\nBest  epoch={best['epoch']}  "
              f"val_psnr={best.get('val',{}).get('val_psnr','n/a'):.4f}  "
              f"val_ssim={best.get('val',{}).get('val_ssim','n/a'):.4f}")
    except ImportError:
        print("matplotlib not available — skipping plot.")
        for h in history[-5:]:
            print(h)


## 🔮 Inference

Loads the **best_infer.pt** checkpoint (or best.pt / latest_infer.pt as fallback)  
and runs predictions on the test set.

Quality presets:

| Preset | TTA | DC Refine Steps | Notes |
|--------|-----|-----------------|-------|
| `fast` | none | 0 | Fastest, good baseline |
| `balanced` | flip4 | 3 | +~0.1 dB typical |
| `high` | flip8 | 8 | Best quality, 8× slower |

Set `QUALITY_PRESET` in the Configuration cell before running.


In [ ]:
import subprocess, sys, os, torch

n_gpus = torch.cuda.device_count()

if n_gpus > 1:
    cmd = [
        "torchrun",
        f"--nproc_per_node={n_gpus}",
        "--master_port=29501",
        "src/infer_nafnet_ddp.py",
        "--config", "infer_config.json",
    ]
    print(f"Inference with torchrun on {n_gpus} GPUs …")
else:
    cmd = [sys.executable, "src/infer_nafnet_ddp.py", "--config", "infer_config.json"]
    print("Inference (single GPU / CPU) …")

print(f"Command: {' '.join(cmd)}\n{'='*70}")

env = os.environ.copy()
env["PYTHONPATH"] = "." + os.pathsep + "nafnet/official" + os.pathsep + env.get("PYTHONPATH", "")

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)
try:
    for line in process.stdout:
        print(line, end="", flush=True)
finally:
    process.wait()

print(f"\n{'='*70}")
if process.returncode == 0:
    import pathlib
    n_pred = len(list(pathlib.Path(SUBMISSION_DIR).glob("*.npy")))
    print(f"✓ Inference complete. {n_pred} predictions written to {SUBMISSION_DIR}")
else:
    print(f"⚠ Inference exited with code {process.returncode}")


## 📤 Build submission.csv

In [ ]:
import subprocess, sys, os

cmd = [
    sys.executable, "src/make_submission_csv.py",
    "--submission-dir", SUBMISSION_DIR,
    "--output-csv",     SUBMISSION_CSV,
    "--expected-size",  str(EXPECTED_SIZE),
]

env = os.environ.copy()
env["PYTHONPATH"] = "." + os.pathsep + env.get("PYTHONPATH", "")

r = subprocess.run(cmd, capture_output=True, text=True, env=env)
print(r.stdout)
if r.returncode != 0:
    print("STDERR:", r.stderr)
    raise RuntimeError("make_submission_csv.py failed.")

import os
size_mb = os.path.getsize(SUBMISSION_CSV) / 1e6
print(f"submission.csv  {size_mb:.1f} MB  →  {SUBMISSION_CSV}")


In [ ]:
import csv, base64, io, numpy as np
from pathlib import Path

csv_path = Path(SUBMISSION_CSV)
assert csv_path.exists(), f"submission.csv not found: {csv_path}"

with open(csv_path, newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

print(f"Rows  : {len(rows)}")

# Decode and spot-check first and last entries
for label, row in [("First", rows[0]), ("Last", rows[-1])]:
    buf = io.BytesIO(base64.b64decode(row["npy_base64"]))
    arr = np.load(buf)
    print(f"{label} (id={row['id']:>4}) : shape={arr.shape}  "
          f"dtype={arr.dtype}  range=[{arr.min():.4f}, {arr.max():.4f}]")
    assert arr.shape == (EXPECTED_SIZE, EXPECTED_SIZE), f"Unexpected shape: {arr.shape}"
    assert np.all(np.isfinite(arr)), "NaN/Inf detected!"

print(f"\n✓ Submission looks valid — ready to submit as {csv_path.name}")
